# Formula 1 podium prediction

## 1. Setup

In [1]:
CLEAN_SOURCE = '"""Cleaning rules for the Formula 1 podium-prediction milestone.\n\nRaw CSVs are read and never written. Sentinels are replaced on copies, types are\napplied, the broken lap-time key is repaired, and a separate driver-race table\nis built with one row per (raceId, driverId).\n\nThis module does not build model features. Columns that are only known after\nthe race are labeled as such in DRIVER_RACE_DICTIONARY.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\nSENTINEL = r"\\N"\nASSIGNMENT_LAST_YEAR = 2024\n\nTABLES = (\n    "circuits",\n    "constructors",\n    "constructor_results",\n    "constructor_standings",\n    "drivers",\n    "driver_standings",\n    "races",\n    "results",\n    "qualifying",\n    "lap_times",\n    "pit_stops",\n    "sprint_results",\n    "seasons",\n    "status",\n)\n\n# Primary keys of the Ergast extracts. lap_times is the only one that fails\n# on the raw files; cleaning makes it unique.\nPRIMARY_KEYS: dict[str, list[str]] = {\n    "circuits": ["circuitId"],\n    "constructors": ["constructorId"],\n    "constructor_results": ["constructorResultsId"],\n    "constructor_standings": ["constructorStandingsId"],\n    "drivers": ["driverId"],\n    "driver_standings": ["driverStandingsId"],\n    "races": ["raceId"],\n    "results": ["resultId"],\n    "qualifying": ["qualifyId"],\n    "lap_times": ["raceId", "driverId", "lap"],\n    "pit_stops": ["raceId", "driverId", "stop"],\n    "sprint_results": ["resultId"],\n    "seasons": ["year"],\n    "status": ["statusId"],\n}\n\n# Grain checks that are not the single-column primary key.\nGRAIN_KEYS: dict[str, list[str]] = {\n    "results_driver_race": ["raceId", "driverId"],\n    "qualifying_driver_race": ["raceId", "driverId"],\n    "sprint_driver_race": ["raceId", "driverId"],\n    "driver_standings_driver_race": ["raceId", "driverId"],\n    "constructor_standings_race": ["raceId", "constructorId"],\n    "constructor_results_race": ["raceId", "constructorId"],\n}\n\nGRAIN_TABLE = {\n    "results_driver_race": "results",\n    "qualifying_driver_race": "qualifying",\n    "sprint_driver_race": "sprint_results",\n    "driver_standings_driver_race": "driver_standings",\n    "constructor_standings_race": "constructor_standings",\n    "constructor_results_race": "constructor_results",\n}\n\n# child table, child column, parent table, parent column\nFOREIGN_KEYS = (\n    ("races", "circuitId", "circuits", "circuitId"),\n    ("results", "raceId", "races", "raceId"),\n    ("results", "driverId", "drivers", "driverId"),\n    ("results", "constructorId", "constructors", "constructorId"),\n    ("results", "statusId", "status", "statusId"),\n    ("qualifying", "raceId", "races", "raceId"),\n    ("qualifying", "driverId", "drivers", "driverId"),\n    ("qualifying", "constructorId", "constructors", "constructorId"),\n    ("driver_standings", "raceId", "races", "raceId"),\n    ("driver_standings", "driverId", "drivers", "driverId"),\n    ("constructor_standings", "raceId", "races", "raceId"),\n    ("constructor_standings", "constructorId", "constructors", "constructorId"),\n    ("constructor_results", "raceId", "races", "raceId"),\n    ("constructor_results", "constructorId", "constructors", "constructorId"),\n    ("lap_times", "raceId", "races", "raceId"),\n    ("lap_times", "driverId", "drivers", "driverId"),\n    ("pit_stops", "raceId", "races", "raceId"),\n    ("pit_stops", "driverId", "drivers", "driverId"),\n    ("sprint_results", "raceId", "races", "raceId"),\n    ("sprint_results", "driverId", "drivers", "driverId"),\n    ("sprint_results", "constructorId", "constructors", "constructorId"),\n    ("sprint_results", "statusId", "status", "statusId"),\n)\n\nINTEGER_COLUMNS: dict[str, list[str]] = {\n    "circuits": ["circuitId"],\n    "constructors": ["constructorId"],\n    "constructor_results": ["constructorResultsId", "raceId", "constructorId"],\n    "constructor_standings": ["constructorStandingsId", "raceId", "constructorId", "position", "wins"],\n    "drivers": ["driverId", "number"],\n    "driver_standings": ["driverStandingsId", "raceId", "driverId", "position", "wins"],\n    "races": ["raceId", "year", "round", "circuitId"],\n    "results": [\n        "resultId", "raceId", "driverId", "constructorId", "number", "grid",\n        "position", "positionOrder", "laps", "milliseconds", "fastestLap", "rank", "statusId",\n    ],\n    "qualifying": ["qualifyId", "raceId", "driverId", "constructorId", "number", "position"],\n    "lap_times": ["raceId", "driverId", "lap", "position", "milliseconds"],\n    "pit_stops": ["raceId", "driverId", "stop", "lap", "milliseconds"],\n    "sprint_results": [\n        "resultId", "raceId", "driverId", "constructorId", "number", "grid",\n        "position", "positionOrder", "laps", "milliseconds", "fastestLap", "statusId", "rank",\n    ],\n    "seasons": ["year"],\n    "status": ["statusId"],\n}\n\nFLOAT_COLUMNS: dict[str, list[str]] = {\n    "circuits": ["lat", "lng", "alt"],\n    "constructor_results": ["points"],\n    "constructor_standings": ["points"],\n    "driver_standings": ["points"],\n    "results": ["points", "fastestLapSpeed"],\n    "sprint_results": ["points"],\n}\n\nDATE_COLUMNS: dict[str, list[str]] = {\n    "drivers": ["dob"],\n    "races": [\n        "date", "fp1_date", "fp2_date", "fp3_date", "quali_date", "sprint_date",\n    ],\n}\n\n# Clock strings such as 06:00:00. Kept as text and also combined with the date.\nCLOCK_COLUMNS: dict[str, list[str]] = {\n    "races": ["time", "fp1_time", "fp2_time", "fp3_time", "quali_time", "sprint_time"],\n}\n\nDURATION_COLUMNS: dict[str, list[str]] = {\n    "qualifying": ["q1", "q2", "q3"],\n    "lap_times": ["time"],\n    "pit_stops": ["duration"],\n    "results": ["fastestLapTime"],\n    "sprint_results": ["fastestLapTime"],\n}\n\nDRIVER_RACE_DICTIONARY = (\n    ("raceId", "identifier", "Race key. Already known when the race is scheduled."),\n    ("driverId", "identifier", "Driver key."),\n    ("resultId", "identifier", "resultId of the row kept by the collapse rule."),\n    ("year", "pre_race_context", "Season. Known before the race."),\n    ("round", "pre_race_context", "Round number. Known before the race."),\n    ("race_date", "pre_race_context", "Scheduled race date."),\n    ("circuitId", "pre_race_context", "Circuit. Known before the race."),\n    ("race_name", "pre_race_context", "Grand Prix name."),\n    ("constructorId", "pre_race_context", "Constructor on the kept row. When n_constructors > 1 the driver drove more than one car, so this is the car of the better classified stint, not necessarily the car they qualified."),\n    ("car_number", "pre_race_context", "Car number on the kept row."),\n    ("grid", "pre_race_context", "Starting grid. 0 means no grid slot (pit lane or did not start). Null means the source file omitted the grid for that row."),\n    ("n_result_rows", "audit", "How many results rows this driver had in this race before the collapse."),\n    ("n_constructors", "audit", "How many constructors those rows used."),\n    ("shared_entry", "audit", "True when n_result_rows > 1. These are shared drives or double entries, almost all from 1950-1964, plus one 1978 case."),\n    ("position", "post_race_label", "Official classified position. Null means not classified. Not a legal model feature: it is known only after the race."),\n    ("positionText", "post_race_label", "Official position text, including R, D, W, F, N. Not a legal model feature."),\n    ("positionOrder", "post_race_label", "Dense rank of every starter, including retirements. The brief defines the label from this column. Not a legal model feature."),\n    ("points", "post_race_label", "Points scored in this race. Not a legal model feature."),\n    ("laps", "post_race_label", "Laps completed. Not a legal model feature."),\n    ("statusId", "post_race_label", "Finishing status. Not a legal model feature."),\n    ("podium", "post_race_label", "Brief\'s label: positionOrder <= 3."),\n    ("official_podium", "post_race_label", "positionText is 1, 2, or 3. Kept so the one known disagreement with podium can be audited."),\n    ("podium_definition_conflict", "audit", "True when podium and official_podium disagree."),\n    ("in_assignment_window", "audit", "True when year <= 2024, the window named in the brief. Later completed races are retained and flagged, not deleted."),\n    ("collapse_rule", "audit", "Rule that chose resultId. The same rule is used on every row."),\n)\n\nCOLLAPSE_RULE = "best_classification"\n\n\n@dataclass\nclass CleaningResult:\n    cleaned: dict[str, pd.DataFrame]\n    sentinel_summary: pd.DataFrame\n    keys_before: pd.DataFrame\n    keys_after: pd.DataFrame\n    coercion_failures: pd.DataFrame\n    lap_conflict_summary: pd.DataFrame\n    lap_conflicts: pd.DataFrame\n    duplicate_rows: pd.DataFrame\n    rule_comparison: pd.DataFrame\n    driver_race: pd.DataFrame\n    races_without_results: pd.DataFrame\n    time_checks: pd.DataFrame\n    missing_policy: pd.DataFrame\n\n\ndef project_paths(root: Path | None = None) -> dict[str, Path]:\n    root = Path(root) if root is not None else Path(__file__).resolve().parents[1]\n    return {\n        "root": root,\n        "raw": root / "data" / "raw",\n        "cleaned": root / "data" / "cleaned",\n        "processed": root / "data" / "processed",\n        "audit": root / "data" / "audit",\n        "figures": root / "figures",\n    }\n\n\ndef file_sha256(path: Path) -> str:\n    digest = hashlib.sha256()\n    with path.open("rb") as handle:\n        for block in iter(lambda: handle.read(1 << 20), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef raw_hashes(raw_dir: Path) -> dict[str, str]:\n    return {name: file_sha256(raw_dir / f"{name}.csv") for name in TABLES}\n\n\ndef load_raw(raw_dir: Path) -> dict[str, pd.DataFrame]:\n    """Read the 14 assignment tables as strings, with no sentinel conversion."""\n    frames = {}\n    for name in TABLES:\n        path = raw_dir / f"{name}.csv"\n        frames[name] = pd.read_csv(path, dtype=str, keep_default_na=False)\n    return frames\n\n\ndef summarize_sentinels(raw: dict[str, pd.DataFrame]) -> pd.DataFrame:\n    rows = []\n    for name, frame in raw.items():\n        for column in frame.columns:\n            sentinel_n = int((frame[column] == SENTINEL).sum())\n            if sentinel_n == 0:\n                continue\n            rows.append({\n                "table": name,\n                "column": column,\n                "sentinel_n": sentinel_n,\n                "rows": len(frame),\n                "sentinel_share": sentinel_n / len(frame),\n            })\n    summary = pd.DataFrame(rows)\n    return summary.sort_values(["sentinel_share", "table", "column"], ascending=[False, True, True]).reset_index(drop=True)\n\n\n# What to do with each column that contains the \\N marker.\n# mean, median, and mode are not used: a blank here means "not recorded" or "did not happen".\nMISSING_POLICY: dict[tuple[str, str], tuple[str, str]] = {\n    ("constructor_results", "status"): (\n        "code as not disqualified",\n        "\\\\N means the constructor was not disqualified. Stored as disqualified=False. D stays True.",\n    ),\n    ("constructor_standings", "position"): (\n        "leave missing",\n        "One standings row is unranked. A median rank would invent a place.",\n    ),\n    ("drivers", "number"): (\n        "leave missing",\n        "Permanent numbers did not exist for most of the history. The mode would copy one modern number onto old drivers.",\n    ),\n    ("drivers", "code"): (\n        "leave missing",\n        "Three-letter codes did not exist for most of the history. The mode would copy one code onto old drivers.",\n    ),\n    ("driver_standings", "position"): (\n        "leave missing",\n        "The text is already \'-\': unranked. The mode would assign a championship place.",\n    ),\n    ("races", "time"): (\n        "leave missing",\n        "The start clock was not stored. An average of clock times is not a start time.",\n    ),\n    ("races", "fp1_date"): ("leave missing", "This session was not stored for that race."),\n    ("races", "fp1_time"): ("leave missing", "This session was not stored for that race."),\n    ("races", "fp2_date"): ("leave missing", "This session was not stored for that race."),\n    ("races", "fp2_time"): ("leave missing", "This session was not stored for that race."),\n    ("races", "fp3_date"): ("leave missing", "This session was not stored for that race."),\n    ("races", "fp3_time"): ("leave missing", "This session was not stored for that race."),\n    ("races", "quali_date"): ("leave missing", "This session was not stored for that race."),\n    ("races", "quali_time"): ("leave missing", "This session was not stored for that race."),\n    ("races", "sprint_date"): ("leave missing", "This race had no sprint, or the sprint date was not stored."),\n    ("races", "sprint_time"): ("leave missing", "This race had no sprint, or the sprint time was not stored."),\n    ("results", "number"): (\n        "leave missing",\n        "A few cars have no number. The mode would give them a number that already belongs to another car.",\n    ),\n    ("results", "grid"): (\n        "leave missing",\n        "The file omitted the grid for two whole races. The median grid would pretend those drivers started from midfield. Grid 0 is a real value and is not filled.",\n    ),\n    ("results", "position"): (\n        "leave missing",\n        "Empty means not classified. positionOrder is already filled. A median position would create a finish.",\n    ),\n    ("results", "time"): (\n        "leave missing",\n        "No finish clock was stored. A median time would give a retired car a finish time. Known only after the race.",\n    ),\n    ("results", "milliseconds"): (\n        "leave missing",\n        "No elapsed time was stored. Same column as the finish clock, in milliseconds.",\n    ),\n    ("results", "fastestLap"): (\n        "leave missing",\n        "No fastest lap was stored. Known only after the race.",\n    ),\n    ("results", "rank"): (\n        "leave missing",\n        "No fastest-lap rank was stored. Known only after the race.",\n    ),\n    ("results", "fastestLapTime"): (\n        "leave missing",\n        "No fastest-lap time was stored. A median lap time would invent one.",\n    ),\n    ("results", "fastestLapSpeed"): (\n        "leave missing",\n        "No fastest-lap speed was stored. The distribution is skewed, and a median would still invent a speed.",\n    ),\n    ("qualifying", "q1"): (\n        "leave missing",\n        "The driver set no Q1 time. A median lap time would invent one.",\n    ),\n    ("qualifying", "q2"): (\n        "leave missing",\n        "The driver did not reach Q2. A median would say they set a Q2 time.",\n    ),\n    ("qualifying", "q3"): (\n        "leave missing",\n        "The driver did not reach Q3. A median would say they set a Q3 time.",\n    ),\n    ("pit_stops", "duration"): (\n        "leave missing",\n        "Three stops have no duration. A median would hide a broken measurement. Pit stops happen during the race.",\n    ),\n    ("pit_stops", "milliseconds"): (\n        "leave missing",\n        "The same three stops have no duration in milliseconds.",\n    ),\n    ("sprint_results", "position"): (\n        "leave missing",\n        "The driver was not classified in the sprint.",\n    ),\n    ("sprint_results", "time"): (\n        "leave missing",\n        "No sprint finish clock was stored.",\n    ),\n    ("sprint_results", "milliseconds"): (\n        "leave missing",\n        "No sprint elapsed time was stored.",\n    ),\n    ("sprint_results", "fastestLap"): (\n        "leave missing",\n        "No sprint fastest lap was stored.",\n    ),\n    ("sprint_results", "fastestLapTime"): (\n        "leave missing",\n        "No sprint fastest-lap time was stored.",\n    ),\n    ("sprint_results", "statusId"): (\n        "leave missing",\n        "Two sprint rows have no status. The mode would mark them as the most common status.",\n    ),\n    ("sprint_results", "rank"): (\n        "leave missing",\n        "No sprint fastest-lap rank was stored.",\n    ),\n}\n\nDEFAULT_MISSING_POLICY = (\n    "leave missing",\n    "The marker is an absent value. Mean, median, and mode would invent one.",\n)\n\n\ndef missing_value_report(raw: dict[str, pd.DataFrame], cleaned: dict[str, pd.DataFrame]) -> pd.DataFrame:\n    """Record the method applied to every column that contained \\\\N. Nothing is imputed."""\n    rows = []\n    for name, frame in raw.items():\n        for column in frame.columns:\n            sentinel_n = int((frame[column] == SENTINEL).sum())\n            if sentinel_n == 0:\n                continue\n            method, reason = MISSING_POLICY.get((name, column), DEFAULT_MISSING_POLICY)\n            missing_after = int(cleaned[name][column].isna().sum())\n            rows.append({\n                "table": name,\n                "column": column,\n                "blank_markers": sentinel_n,\n                "method": method,\n                "values_filled": sentinel_n - missing_after,\n                "still_missing": missing_after,\n                "reason": reason,\n            })\n    status = raw["constructor_results"]["status"]\n    disqualified = cleaned["constructor_results"]["disqualified"]\n    rows.append({\n        "table": "constructor_results",\n        "column": "disqualified",\n        "blank_markers": int((status == SENTINEL).sum()),\n        "method": "filled with False",\n        "values_filled": int((~disqualified).sum()),\n        "still_missing": int(disqualified.isna().sum()),\n        "reason": "Each \\\\N in status was written here as False. Every row now has True or False.",\n    })\n    report = pd.DataFrame(rows)\n    if report.empty:\n        return report\n    return report.sort_values(["table", "column"]).reset_index(drop=True)\n\n\ndef replace_sentinels(frame: pd.DataFrame) -> pd.DataFrame:\n    """Turn the Ergast null marker into missing values. Does not change row count."""\n    cleaned = frame.copy()\n    return cleaned.mask(cleaned == SENTINEL, pd.NA)\n\n\ndef parse_duration_to_seconds(series: pd.Series) -> tuple[pd.Series, int]:\n    """Parse ss.sss, m:ss.sss, or h:mm:ss.sss. A leading + is removed by the caller when needed.\n\n    Returns the seconds and the number of non-missing values that did not parse.\n    """\n    text = series.astype("string").str.strip()\n    missing = text.isna() | (text == "")\n    n_colons = text.str.count(":").fillna(0)\n    parts = text.str.split(":", n=2, expand=True)\n    while parts.shape[1] < 3:\n        parts[parts.shape[1]] = pd.NA\n    numbers = [pd.to_numeric(parts[i], errors="coerce") for i in range(3)]\n    seconds = pd.Series(np.nan, index=series.index, dtype="float64")\n    one = (~missing) & (n_colons == 0)\n    two = (~missing) & (n_colons == 1)\n    three = (~missing) & (n_colons == 2)\n    seconds.loc[one] = numbers[0].loc[one]\n    seconds.loc[two] = numbers[0].loc[two] * 60 + numbers[1].loc[two]\n    seconds.loc[three] = numbers[0].loc[three] * 3600 + numbers[1].loc[three] * 60 + numbers[2].loc[three]\n    failed = int((~missing & seconds.isna()).sum())\n    return seconds, failed\n\n\ndef _cast_numeric(series: pd.Series, dtype: str) -> tuple[pd.Series, int]:\n    missing = series.isna()\n    numeric = pd.to_numeric(series, errors="coerce")\n    failed = ~missing & numeric.isna()\n    if dtype == "Int64":\n        fractional = numeric.notna() & ((numeric % 1) != 0)\n        failed = failed | fractional\n        numeric = numeric.mask(fractional)\n        numeric = numeric.astype("Int64")\n    else:\n        numeric = numeric.astype("float64")\n    return numeric, int(failed.sum())\n\n\ndef _cast_dates(series: pd.Series) -> tuple[pd.Series, int]:\n    missing = series.isna()\n    parsed = pd.to_datetime(series, format="%Y-%m-%d", errors="coerce")\n    failed = int((~missing & parsed.isna()).sum())\n    return parsed, failed\n\n\ndef _record_failure(rows: list[dict], table: str, column: str, failed: int) -> None:\n    if failed:\n        rows.append({"table": table, "column": column, "failed_n": failed})\n\n\ndef _apply_declared_types(name: str, frame: pd.DataFrame, failures: list[dict]) -> pd.DataFrame:\n    """Cast the columns this file actually has. A missing optional column is skipped.\n\n    The Kaggle extract does not always include `rank` (fastest-lap rank). That\n    field is only known after the race, so its absence does not block cleaning.\n    """\n    cleaned = frame.copy()\n    for column in INTEGER_COLUMNS.get(name, []):\n        if column not in cleaned.columns:\n            continue\n        cleaned[column], failed = _cast_numeric(cleaned[column], "Int64")\n        _record_failure(failures, name, column, failed)\n    for column in FLOAT_COLUMNS.get(name, []):\n        if column not in cleaned.columns:\n            continue\n        cleaned[column], failed = _cast_numeric(cleaned[column], "float64")\n        _record_failure(failures, name, column, failed)\n    for column in DATE_COLUMNS.get(name, []):\n        if column not in cleaned.columns:\n            continue\n        cleaned[column], failed = _cast_dates(cleaned[column])\n        _record_failure(failures, name, column, failed)\n    for column in DURATION_COLUMNS.get(name, []):\n        if column not in cleaned.columns:\n            continue\n        seconds, failed = parse_duration_to_seconds(cleaned[column])\n        cleaned[f"{column}_seconds"] = seconds\n        _record_failure(failures, name, f"{column}_seconds", failed)\n    return cleaned\n\n\ndef _parse_results_time(frame: pd.DataFrame, table: str, failures: list[dict]) -> pd.DataFrame:\n    """`time` mixes two different measurements. Keep them in separate columns.\n\n    A value with no leading + is the winner\'s elapsed race time.\n    A value with a leading + is the gap to the winner.\n    `milliseconds`, when present, is elapsed time for that car, including cars\n    whose `time` string is only a gap. It is not the gap itself.\n    """\n    cleaned = frame.copy()\n    if "time" not in cleaned.columns:\n        return cleaned\n    text = cleaned["time"].astype("string")\n    is_gap = text.str.startswith("+", na=False)\n    gap_seconds, gap_failed = parse_duration_to_seconds(text.where(is_gap).str.replace(r"^\\+", "", regex=True))\n    winner_seconds, winner_failed = parse_duration_to_seconds(text.where(~is_gap))\n    cleaned["gap_seconds"] = gap_seconds\n    cleaned["winner_time_seconds"] = winner_seconds\n    _record_failure(failures, table, "gap_seconds", gap_failed)\n    _record_failure(failures, table, "winner_time_seconds", winner_failed)\n    if "milliseconds" in cleaned.columns:\n        cleaned["elapsed_seconds"] = cleaned["milliseconds"].astype("Float64") / 1000\n    return cleaned\n\n\ndef _add_race_start(frame: pd.DataFrame, failures: list[dict]) -> pd.DataFrame:\n    cleaned = frame.copy()\n    date_text = cleaned["date"].dt.strftime("%Y-%m-%d")\n    clock = cleaned["time"].astype("string")\n    both = date_text.notna() & clock.notna()\n    stamp = pd.to_datetime(date_text.where(both) + " " + clock.where(both), errors="coerce")\n    failed = int((both & stamp.isna()).sum())\n    _record_failure(failures, "races", "race_start", failed)\n    cleaned["race_start"] = stamp\n    return cleaned\n\n\ndef _key_rows_for(name: str, frame: pd.DataFrame, keys: list[str], check: str) -> dict:\n    missing = frame[keys].isna().any(axis=1)\n    problem_rows = int(frame.duplicated(keys).sum())\n    return {\n        "table": name,\n        "check": check,\n        "columns": ", ".join(keys),\n        "rows": len(frame),\n        "null_key_rows": int(missing.sum()),\n        "problem_rows": problem_rows,\n        "passed": bool(missing.sum() == 0 and problem_rows == 0),\n    }\n\n\ndef audit_keys(frames: dict[str, pd.DataFrame]) -> pd.DataFrame:\n    rows = []\n    for name, keys in PRIMARY_KEYS.items():\n        rows.append(_key_rows_for(name, frames[name], keys, "primary_key"))\n    for grain_name, keys in GRAIN_KEYS.items():\n        table = GRAIN_TABLE[grain_name]\n        rows.append(_key_rows_for(grain_name, frames[table], keys, "grain"))\n    parent_ids = {name: set(frame[PRIMARY_KEYS[name][0]].dropna()) for name, frame in frames.items() if len(PRIMARY_KEYS[name]) == 1}\n    for child, column, parent, parent_column in FOREIGN_KEYS:\n        values = frames[child][column]\n        missing = values.isna()\n        orphans = ~missing & ~values.isin(parent_ids[parent])\n        rows.append({\n            "table": child,\n            "check": "foreign_key",\n            "columns": f"{column} -> {parent}.{parent_column}",\n            "rows": len(frames[child]),\n            "null_key_rows": int(missing.sum()),\n            "problem_rows": int(orphans.sum()),\n            "passed": bool(orphans.sum() == 0),\n        })\n    return pd.DataFrame(rows)\n\n\ndef deduplicate_lap_times(frame: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """Make (raceId, driverId, lap) unique.\n\n    Exact copies are dropped. When two copies disagree, the faster time is kept\n    and both measurements are written to the conflict audit. Lap times happen\n    during the race, so this repair is for key integrity, not for model features.\n    """\n    keys = ["raceId", "driverId", "lap"]\n    ordered = frame.sort_values(keys + ["milliseconds"], kind="mergesort")\n    grouped = ordered.groupby(keys, dropna=False)["milliseconds"]\n    spread = grouped.agg(n="size", min_ms="min", max_ms="max").reset_index()\n    summary = pd.DataFrame([\n        {"kind": "unique_already", "groups": int((spread["n"] == 1).sum())},\n        {"kind": "exact_duplicate", "groups": int(((spread["n"] > 1) & (spread["min_ms"] == spread["max_ms"])).sum())},\n        {"kind": "one_millisecond", "groups": int(((spread["n"] > 1) & ((spread["max_ms"] - spread["min_ms"]) == 1)).sum())},\n        {"kind": "larger_conflict", "groups": int(((spread["n"] > 1) & ((spread["max_ms"] - spread["min_ms"]) > 1)).sum())},\n    ])\n    conflict_keys = spread.loc[spread["n"] > 1, keys]\n    conflict_rows = ordered.merge(conflict_keys, on=keys, how="inner")\n    conflict_rows = conflict_rows.copy()\n    conflict_rows["delta_ms"] = conflict_rows.groupby(keys)["milliseconds"].transform(lambda s: s.max() - s.min())\n    kept = ordered.drop_duplicates(keys, keep="first").reset_index(drop=True)\n    return kept, summary, conflict_rows.reset_index(drop=True)\n\n\ndef collapse_entries(results: pd.DataFrame, rule: str) -> pd.DataFrame:\n    """Keep one results row per driver per race.\n\n    best_classification: lowest positionOrder, then most laps, then smallest resultId.\n    most_laps: most laps, then lowest positionOrder, then smallest resultId.\n    """\n    if rule == "best_classification":\n        sort_cols = ["raceId", "driverId", "positionOrder", "laps", "resultId"]\n        ascending = [True, True, True, False, True]\n    elif rule == "most_laps":\n        sort_cols = ["raceId", "driverId", "laps", "positionOrder", "resultId"]\n        ascending = [True, True, False, True, True]\n    else:\n        raise ValueError(f"Unknown collapse rule: {rule}")\n    ordered = results.sort_values(sort_cols, ascending=ascending, kind="mergesort")\n    return ordered.drop_duplicates(["raceId", "driverId"], keep="first")\n\n\ndef compare_collapse_rules(results: pd.DataFrame) -> pd.DataFrame:\n    """Run both candidate rules on duplicate pairs only."""\n    duplicate = results["n_result_rows"] > 1 if "n_result_rows" in results.columns else results.duplicated(["raceId", "driverId"], keep=False)\n    subset = results.loc[duplicate].copy()\n    if "n_result_rows" not in subset.columns:\n        subset["n_result_rows"] = subset.groupby(["raceId", "driverId"])["resultId"].transform("size")\n    best = collapse_entries(subset, "best_classification")\n    laps = collapse_entries(subset, "most_laps")\n    compared = best[["raceId", "driverId", "resultId", "positionOrder", "laps", "constructorId", "grid"]].merge(\n        laps[["raceId", "driverId", "resultId", "positionOrder", "laps", "constructorId", "grid"]],\n        on=["raceId", "driverId"],\n        suffixes=("_best", "_laps"),\n    )\n    compared["same_row"] = compared["resultId_best"] == compared["resultId_laps"]\n    compared["same_position"] = compared["positionOrder_best"] == compared["positionOrder_laps"]\n    compared["same_podium"] = (compared["positionOrder_best"] <= 3) == (compared["positionOrder_laps"] <= 3)\n    return compared\n\n\ndef build_driver_race(results: pd.DataFrame, races: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:\n    """One row per driver per race.\n\n    Rule, in order: better classified result (lower positionOrder), then more\n    laps, then the smaller resultId. positionOrder is first because the label\n    is the driver\'s finishing result. The most-laps rule is computed as a check,\n    not as the rule that is stored.\n    """\n    working = results.copy()\n    working["n_result_rows"] = working.groupby(["raceId", "driverId"])["resultId"].transform("size")\n    working["n_constructors"] = working.groupby(["raceId", "driverId"])["constructorId"].transform("nunique")\n    comparison = compare_collapse_rules(working)\n    kept = collapse_entries(working, COLLAPSE_RULE).copy()\n    kept["shared_entry"] = kept["n_result_rows"] > 1\n    kept["podium"] = kept["positionOrder"] <= 3\n    kept["official_podium"] = kept["positionText"].isin(["1", "2", "3"])\n    kept["podium_definition_conflict"] = kept["podium"] != kept["official_podium"]\n    kept["collapse_rule"] = COLLAPSE_RULE\n\n    context = races[["raceId", "year", "round", "date", "circuitId", "name"]].rename(\n        columns={"date": "race_date", "name": "race_name"}\n    )\n    driver_race = kept.merge(context, on="raceId", how="left", validate="many_to_one")\n    driver_race["in_assignment_window"] = driver_race["year"] <= ASSIGNMENT_LAST_YEAR\n    driver_race = driver_race.rename(columns={"number": "car_number"})\n    columns = [name for name, _, _ in DRIVER_RACE_DICTIONARY]\n    driver_race = driver_race[columns].sort_values(["year", "round", "positionOrder", "driverId"]).reset_index(drop=True)\n\n    duplicate_rows = working.loc[working["n_result_rows"] > 1].copy()\n    duplicate_rows["kept"] = duplicate_rows["resultId"].isin(set(driver_race["resultId"]))\n    return driver_race, duplicate_rows, comparison\n\n\ndef time_consistency(cleaned: dict[str, pd.DataFrame]) -> pd.DataFrame:\n    """Parsed strings should match the millisecond columns already in the file."""\n    rows = []\n\n    results = cleaned["results"]\n    absolute = results["winner_time_seconds"].notna() & results["elapsed_seconds"].notna()\n    if absolute.any():\n        delta = (results.loc[absolute, "winner_time_seconds"] - results.loc[absolute, "elapsed_seconds"]).abs()\n        rows.append({"check": "results winner time vs milliseconds", "rows": int(absolute.sum()), "max_abs_seconds": float(delta.max())})\n\n    laps = cleaned["lap_times"]\n    lap_delta = (laps["time_seconds"] - laps["milliseconds"].astype("float64") / 1000).abs()\n    rows.append({"check": "lap time string vs milliseconds", "rows": int(lap_delta.notna().sum()), "max_abs_seconds": float(lap_delta.max())})\n\n    pits = cleaned["pit_stops"]\n    known = pits["duration_seconds"].notna() & pits["milliseconds"].notna()\n    pit_delta = (pits.loc[known, "duration_seconds"] - pits.loc[known, "milliseconds"].astype("float64") / 1000).abs()\n    rows.append({\n        "check": "pit duration string vs milliseconds",\n        "rows": int(known.sum()),\n        "max_abs_seconds": float(pit_delta.max()) if known.any() else 0.0,\n    })\n    return pd.DataFrame(rows)\n\n\ndef clean_tables(raw: dict[str, pd.DataFrame]) -> CleaningResult:\n    keys_before = audit_keys(raw)\n    sentinel_summary = summarize_sentinels(raw)\n    failures: list[dict] = []\n    cleaned: dict[str, pd.DataFrame] = {}\n    for name, frame in raw.items():\n        typed = _apply_declared_types(name, replace_sentinels(frame), failures)\n        if name in {"results", "sprint_results"}:\n            typed = _parse_results_time(typed, name, failures)\n        if name == "races":\n            typed = _add_race_start(typed, failures)\n        if name == "constructor_results":\n            # In this column \\N means "not disqualified", not "status unknown".\n            typed["disqualified"] = typed["status"].eq("D")\n        if name == "results":\n            typed["classified"] = typed["position"].notna()\n        cleaned[name] = typed\n\n    cleaned["lap_times"], lap_summary, lap_conflicts = deduplicate_lap_times(cleaned["lap_times"])\n    driver_race, duplicate_rows, comparison = build_driver_race(cleaned["results"], cleaned["races"])\n    keys_after = audit_keys(cleaned)\n    # The modeling grain is a new table, so add its key check beside the source-table audit.\n    grain = _key_rows_for("driver_race", driver_race, ["raceId", "driverId"], "grain")\n    keys_after = pd.concat([keys_after, pd.DataFrame([grain])], ignore_index=True)\n\n    result_races = set(cleaned["results"]["raceId"])\n    missing_results = cleaned["races"].loc[~cleaned["races"]["raceId"].isin(result_races), ["raceId", "year", "round", "name", "date"]].copy()\n    missing_results = missing_results.sort_values(["year", "round"]).reset_index(drop=True)\n\n    return CleaningResult(\n        cleaned=cleaned,\n        sentinel_summary=sentinel_summary,\n        keys_before=keys_before,\n        keys_after=keys_after,\n        coercion_failures=pd.DataFrame(failures, columns=["table", "column", "failed_n"]),\n        lap_conflict_summary=lap_summary,\n        lap_conflicts=lap_conflicts,\n        duplicate_rows=duplicate_rows,\n        rule_comparison=comparison,\n        driver_race=driver_race,\n        races_without_results=missing_results,\n        time_checks=time_consistency(cleaned),\n        missing_policy=missing_value_report(raw, cleaned),\n    )\n\n\ndef save_outputs(result: CleaningResult, paths: dict[str, Path]) -> None:\n    for folder in ("cleaned", "processed", "audit"):\n        paths[folder].mkdir(parents=True, exist_ok=True)\n    for name, frame in result.cleaned.items():\n        frame.to_csv(paths["cleaned"] / f"{name}.csv", index=False)\n    result.driver_race.to_csv(paths["processed"] / "driver_race.csv", index=False)\n    dictionary = pd.DataFrame(DRIVER_RACE_DICTIONARY, columns=["column", "role", "meaning"])\n    dictionary.to_csv(paths["processed"] / "driver_race_dictionary.csv", index=False)\n    result.sentinel_summary.to_csv(paths["audit"] / "sentinel_summary.csv", index=False)\n    result.keys_before.to_csv(paths["audit"] / "keys_before.csv", index=False)\n    result.keys_after.to_csv(paths["audit"] / "keys_after.csv", index=False)\n    result.coercion_failures.to_csv(paths["audit"] / "coercion_failures.csv", index=False)\n    result.lap_conflict_summary.to_csv(paths["audit"] / "lap_time_conflict_summary.csv", index=False)\n    result.lap_conflicts.to_csv(paths["audit"] / "lap_time_conflicts.csv", index=False)\n    result.duplicate_rows.to_csv(paths["audit"] / "duplicate_result_rows.csv", index=False)\n    result.rule_comparison.to_csv(paths["audit"] / "collapse_rule_comparison.csv", index=False)\n    result.races_without_results.to_csv(paths["audit"] / "races_without_results.csv", index=False)\n    result.time_checks.to_csv(paths["audit"] / "time_checks.csv", index=False)\n    result.missing_policy.to_csv(paths["audit"] / "missing_value_policy.csv", index=False)\n'
FEATURES_SOURCE = '"""Leakage-safe features for one driver in one race.\n\nThe prediction cutoff is after qualifying and before the race starts.\nCurrent-race results, points, laps, status, pit stops, and the standings\npublished for this race are not features. Rolling form and entering\nstandings use only earlier races.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\n# First season in which every completed race in this extract has qualifying.\nQUALIFYING_COMPLETE_FROM = 2003\nTRAIN_LAST_YEAR = 2018\nVALIDATION_LAST_YEAR = 2021\nTEST_LAST_YEAR = 2024\n\nFEATURE_DICTIONARY = (\n    ("grid", "grid", "Starting grid after penalties. Known before the race. 0 means no grid slot, and it is not filled in."),\n    ("front_row", "grid", "1 when the grid is 1 or 2, otherwise 0. Missing when the grid itself is missing."),\n    ("quali_position", "qualifying", "Official qualifying position for this race. Known after qualifying. Missing when that session is not in the file."),\n    ("reached_q3", "qualifying", "1 when the driver set a Q3 time. Missing for a whole race that did not use Q3, so a blank is not treated as an elimination."),\n    ("quali_position_missing", "qualifying", "1 when quali_position is missing, so an imputed position is not mistaken for a real one."),\n    ("driver_podium_rate_last5", "form", "Share of this driver\'s previous 5 starts that were podiums. This race is excluded."),\n    ("driver_points_last5", "form", "Points from this driver\'s previous 5 starts. This race\'s points are excluded."),\n    ("constructor_podium_rate_last5", "form", "Mean podium share of this constructor\'s cars over its previous 5 races. This race is excluded."),\n    ("career_starts", "form", "Number of race starts this driver had before this one."),\n    ("form_missing", "form", "1 on a driver\'s first start in the file, when the previous-5 rates do not exist."),\n    ("driver_standing_entering", "standings", "Driver championship position after the previous round. The standings row for this race is not used."),\n    ("constructor_standing_entering", "standings", "Constructor championship position after the previous round."),\n    ("standing_missing", "standings", "1 when there is no driver standings row for the previous round. That includes every round 1."),\n    ("season_round", "standings", "Round number in the season. It is known when the calendar is published."),\n    ("home_race", "home", "1 when the circuit\'s country matches the driver\'s nationality. A dual national matches either country."),\n)\n\nFEATURE_COLUMNS = [name for name, _, _ in FEATURE_DICTIONARY]\nFEATURE_GROUPS: dict[str, list[str]] = {}\nfor _name, _group, _meaning in FEATURE_DICTIONARY:\n    FEATURE_GROUPS.setdefault(_group, []).append(_name)\n\n# Post-race fields. They may be used to build a shifted feature, and they must\n# not appear as columns the model is allowed to see.\nLEAKAGE_COLUMNS = (\n    "position",\n    "positionText",\n    "positionOrder",\n    "points",\n    "laps",\n    "statusId",\n    "fastestLap",\n    "rank",\n    "milliseconds",\n    "official_podium",\n)\n\n# Nationality strings in drivers.csv, after stripping whitespace.\n# Circuit countries keep the file\'s own names, including both "USA" and "United States".\nNATIONALITY_COUNTRIES = {\n    "American": ("USA", "United States"),\n    "American-Italian": ("USA", "United States", "Italy"),\n    "Argentine": ("Argentina",),\n    "Argentine-Italian": ("Argentina", "Italy"),\n    "Argentinian": ("Argentina",),\n    "Australian": ("Australia",),\n    "Austrian": ("Austria",),\n    "Belgian": ("Belgium",),\n    "Brazilian": ("Brazil",),\n    "British": ("UK",),\n    "Canadian": ("Canada",),\n    "Chilean": ("Chile",),\n    "Chinese": ("China",),\n    "Colombian": ("Colombia",),\n    "Czech": ("Czech Republic",),\n    "Danish": ("Denmark",),\n    "Dutch": ("Netherlands",),\n    "East German": ("Germany",),\n    "Finnish": ("Finland",),\n    "French": ("France",),\n    "German": ("Germany",),\n    "Hungarian": ("Hungary",),\n    "Indian": ("India",),\n    "Indonesian": ("Indonesia",),\n    "Irish": ("Ireland",),\n    "Italian": ("Italy",),\n    "Japanese": ("Japan",),\n    "Liechtensteiner": ("Liechtenstein",),\n    "Malaysian": ("Malaysia",),\n    "Mexican": ("Mexico",),\n    "Monegasque": ("Monaco",),\n    "New Zealander": ("New Zealand",),\n    "Polish": ("Poland",),\n    "Portuguese": ("Portugal",),\n    "Rhodesian": ("Rhodesia",),\n    "Russian": ("Russia",),\n    "South African": ("South Africa",),\n    "Spanish": ("Spain",),\n    "Swedish": ("Sweden",),\n    "Swiss": ("Switzerland",),\n    "Thai": ("Thailand",),\n    "Uruguayan": ("Uruguay",),\n    "Venezuelan": ("Venezuela",),\n}\n\n\ndef feature_dictionary_frame() -> pd.DataFrame:\n    return pd.DataFrame(FEATURE_DICTIONARY, columns=["column", "group", "meaning"])\n\n\ndef assign_split(year: pd.Series) -> pd.Series:\n    """Chronological split. The test years are never used to choose a model."""\n    split = pd.Series("excluded", index=year.index, dtype="object")\n    split.loc[year.between(QUALIFYING_COMPLETE_FROM, TRAIN_LAST_YEAR)] = "train"\n    split.loc[year.between(TRAIN_LAST_YEAR + 1, VALIDATION_LAST_YEAR)] = "validation"\n    split.loc[year.between(VALIDATION_LAST_YEAR + 1, TEST_LAST_YEAR)] = "test"\n    return split\n\n\ndef _countries_for(nationality: str) -> set[str]:\n    return set(NATIONALITY_COUNTRIES.get(str(nationality).strip(), ()))\n\n\ndef add_home_race(frame: pd.DataFrame, drivers: pd.DataFrame, circuits: pd.DataFrame) -> pd.DataFrame:\n    """Mark a home race from driver nationality and circuit country."""\n    out = frame.copy()\n    driver_nation = drivers[["driverId", "nationality"]].copy()\n    driver_nation["nationality"] = driver_nation["nationality"].astype("string").str.strip()\n    circuit_country = circuits[["circuitId", "country"]].copy()\n    circuit_country["country"] = circuit_country["country"].astype("string").str.strip()\n    out = out.merge(driver_nation, on="driverId", how="left", validate="many_to_one")\n    out = out.merge(circuit_country, on="circuitId", how="left", validate="many_to_one")\n    out["home_race"] = [\n        int(country in _countries_for(nation))\n        for nation, country in zip(out["nationality"], out["country"])\n    ]\n    return out\n\n\ndef _prior_driver_form(frame: pd.DataFrame) -> pd.DataFrame:\n    ordered = frame.sort_values(["driverId", "race_date", "season_round", "raceId"]).copy()\n    podium = ordered["podium"].astype(float)\n    points = ordered["points"].astype(float)\n    ordered["prior_points"] = points.groupby(ordered["driverId"], sort=False).shift(1)\n    ordered["prior_podium"] = podium.groupby(ordered["driverId"], sort=False).shift(1)\n    by_driver = ordered.groupby("driverId", sort=False)\n    ordered["driver_podium_rate_last5"] = by_driver["prior_podium"].transform(\n        lambda s: s.rolling(5, min_periods=1).mean()\n    )\n    ordered["driver_points_last5"] = by_driver["prior_points"].transform(\n        lambda s: s.rolling(5, min_periods=1).sum()\n    )\n    ordered["career_starts"] = by_driver.cumcount()\n    ordered["form_missing"] = ordered["driver_podium_rate_last5"].isna().astype(int)\n    return ordered\n\n\ndef _prior_constructor_form(frame: pd.DataFrame) -> pd.DataFrame:\n    constructor_race = (\n        frame.groupby(["constructorId", "raceId"], as_index=False)\n        .agg(\n            podium_mean=("podium", "mean"),\n            race_date=("race_date", "min"),\n            season_round=("season_round", "min"),\n        )\n        .sort_values(["constructorId", "race_date", "season_round", "raceId"])\n    )\n    prior = constructor_race.groupby("constructorId", sort=False)["podium_mean"].shift(1)\n    constructor_race["constructor_podium_rate_last5"] = prior.groupby(\n        constructor_race["constructorId"], sort=False\n    ).transform(lambda s: s.rolling(5, min_periods=1).mean())\n    return frame.merge(\n        constructor_race[["constructorId", "raceId", "constructor_podium_rate_last5"]],\n        on=["constructorId", "raceId"],\n        how="left",\n        validate="many_to_one",\n    )\n\n\ndef _entering_standings(frame: pd.DataFrame, standings: pd.DataFrame, races: pd.DataFrame, column: str, key: str) -> pd.Series:\n    """Championship position after the previous round, aligned to `frame`."""\n    context = races[["raceId", "year", "round"]].rename(columns={"round": "standings_round"})\n    table = standings[["raceId", key, "position"]].merge(context, on="raceId", how="left")\n    table = table.rename(columns={"position": column, "standings_round": "previous_round"})\n    table["year"] = table["year"].astype("Int64")\n    table["previous_round"] = table["previous_round"].astype("Int64")\n    lookup = table[["year", "previous_round", key, column]].drop_duplicates(["year", "previous_round", key])\n    merged = frame[["year", "season_round", key]].copy()\n    merged["year"] = merged["year"].astype("Int64")\n    merged["previous_round"] = (merged["season_round"] - 1).astype("Int64")\n    merged = merged.merge(lookup, on=["year", "previous_round", key], how="left")\n    return pd.Series(merged[column].to_numpy(), index=frame.index)\n\n\ndef _add_qualifying(frame: pd.DataFrame, qualifying: pd.DataFrame) -> pd.DataFrame:\n    quali = qualifying[["raceId", "driverId", "position", "q3"]].rename(columns={"position": "quali_position"})\n    out = frame.merge(quali, on=["raceId", "driverId"], how="left", validate="one_to_one")\n    race_has_q3 = out.groupby("raceId")["q3"].transform(lambda s: bool(s.notna().any()))\n    reached = pd.Series(np.nan, index=out.index, dtype="float64")\n    reached.loc[race_has_q3] = out.loc[race_has_q3, "q3"].notna().astype(float)\n    out["reached_q3"] = reached\n    out["quali_position_missing"] = out["quali_position"].isna().astype(int)\n    return out.drop(columns=["q3"])\n\n\ndef build_model_frame(driver_race: pd.DataFrame, cleaned: dict[str, pd.DataFrame]) -> pd.DataFrame:\n    """One row per driver per race, with pre-race features and the podium label."""\n    frame = driver_race.copy()\n    frame["season_round"] = frame["round"].astype("Int64")\n    frame = _prior_driver_form(frame)\n    frame = _prior_constructor_form(frame)\n    frame["driver_standing_entering"] = _entering_standings(\n        frame, cleaned["driver_standings"], cleaned["races"], "driver_standing_entering", "driverId"\n    )\n    frame["constructor_standing_entering"] = _entering_standings(\n        frame, cleaned["constructor_standings"], cleaned["races"], "constructor_standing_entering", "constructorId"\n    )\n    frame["standing_missing"] = frame["driver_standing_entering"].isna().astype(int)\n    frame = _add_qualifying(frame, cleaned["qualifying"])\n    known_grid = frame["grid"].notna()\n    frame["front_row"] = pd.Series(np.nan, index=frame.index, dtype="float64")\n    frame.loc[known_grid, "front_row"] = frame.loc[known_grid, "grid"].isin([1, 2]).astype(float)\n    frame = add_home_race(frame, cleaned["drivers"], cleaned["circuits"])\n    frame["split"] = assign_split(frame["year"])\n    keep = [\n        "raceId", "driverId", "year", "race_date", "race_name",\n        "circuitId", "constructorId", "podium", "split", "nationality", "country",\n        *FEATURE_COLUMNS,\n    ]\n    model = frame[keep].sort_values(["year", "season_round", "raceId", "driverId"]).reset_index(drop=True)\n    leaked = [column for column in LEAKAGE_COLUMNS if column in model.columns]\n    if leaked:\n        raise RuntimeError(f"Model frame still has post-race columns: {leaked}")\n    return model\n\n\ndef modeling_sample(frame: pd.DataFrame) -> pd.DataFrame:\n    """Rows where the cutoff is observable: qualifying exists and the year is in the brief."""\n    return frame.loc[frame["split"].isin(["train", "validation", "test"])].copy()\n\n\ndef feature_evidence(frame: pd.DataFrame) -> dict[str, pd.DataFrame]:\n    """Podium rates that motivate the feature groups. Computed on the training years only."""\n    sample = frame.loc[frame["split"] == "train"].copy()\n    sample["podium"] = sample["podium"].astype(int)\n\n    def _rate(table: pd.DataFrame, key: str) -> pd.DataFrame:\n        grouped = table.groupby(key, dropna=False, observed=True)["podium"].agg(starts="size", podium_rate="mean")\n        return grouped.reset_index()\n\n    grid = sample.loc[sample["grid"].notna() & (sample["grid"] > 0)].copy()\n    grid["grid"] = grid["grid"].astype(int)\n    grid_rate = _rate(grid, "grid")\n    grid_rate = grid_rate.loc[grid_rate["starts"] >= 30]\n\n    quali = sample.loc[sample["quali_position"].notna()].copy()\n    quali["quali_position"] = quali["quali_position"].astype(int)\n    quali_rate = _rate(quali, "quali_position")\n    quali_rate = quali_rate.loc[quali_rate["starts"] >= 30]\n\n    standing = sample.loc[sample["driver_standing_entering"].notna()].copy()\n    standing["standing_bin"] = pd.cut(\n        standing["driver_standing_entering"].astype(float),\n        bins=[0, 1, 3, 10, 40],\n        labels=["1", "2-3", "4-10", "11+"],\n    )\n    standing_rate = _rate(standing, "standing_bin")\n\n    form = sample.loc[sample["driver_podium_rate_last5"].notna()].copy()\n    form["form_bin"] = pd.cut(\n        form["driver_podium_rate_last5"],\n        bins=[-0.01, 0, 0.2, 0.4, 1],\n        labels=["0", "0-20%", "20-40%", "40%+"],\n    )\n    form_rate = _rate(form, "form_bin")\n    return {\n        "grid": grid_rate,\n        "qualifying": quali_rate,\n        "standings": standing_rate,\n        "form": form_rate,\n    }\n'
QUESTIONS_SOURCE = '"""The three data-engineering questions in the brief.\n\nEach answer joins more than one table. The grain, the missing rows, and the\npre-race cutoff are part of the result, not only the chart.\n"""\n\nfrom __future__ import annotations\n\nimport numpy as np\nimport pandas as pd\n\nfrom src.f1_features import TEST_LAST_YEAR, add_home_race\n\nNAVY = "#1f4e79"\nCLAY = "#c46b2c"\nTEAL = "#2f6f6a"\nMIN_FRONT_ROW_STARTS = 20\nMIN_CONSTRUCTOR_STARTS = 30\n\n# A start that did not happen. These rows are not in the retirement denominator.\nDID_NOT_START = {\n    "Did not qualify",\n    "Did not prequalify",\n    "Withdrew",\n    "107% Rule",\n}\n\n# The file attributed the end of the race to a crash or to contact.\nCRASH = {\n    "Accident",\n    "Collision",\n    "Collision damage",\n    "Spun off",\n    "Fatal accident",\n    "Debris",\n    "Damage",\n}\n\n# The file attributed the end of the race to the driver, not to a part.\nDRIVER_CAUSE = {\n    "Injured",\n    "Injury",\n    "Physical",\n    "Driver unwell",\n    "Eye injury",\n    "Illness",\n    "Stalled",\n}\n\n# A regulation decision, not a broken component.\nREGULATION = {\n    "Disqualified",\n    "Excluded",\n    "Underweight",\n}\n\n# Not a classified finish, and not evidence of a specific failed component.\nNOT_ATTRIBUTED = {\n    "Retired",\n    "Not classified",\n    "Not restarted",\n    "Handling",\n    "Out of fuel",\n    "Fuel",\n    "Safety",\n    "Safety concerns",\n    "Refuelling",\n    "Fuel rig",\n}\n\n\ndef status_category(name: str) -> str:\n    """Put every finishing status into one bucket used by question 3."""\n    if name == "Finished" or str(name).startswith("+"):\n        return "classified"\n    if name in DID_NOT_START:\n        return "did_not_start"\n    if name in CRASH:\n        return "crash"\n    if name in DRIVER_CAUSE:\n        return "driver"\n    if name in REGULATION:\n        return "regulation"\n    if name in NOT_ATTRIBUTED:\n        return "not_attributed"\n    return "mechanical"\n\n\ndef status_audit(status: pd.DataFrame) -> pd.DataFrame:\n    table = status[["statusId", "status"]].copy()\n    table["category"] = table["status"].map(status_category)\n    return table.sort_values("statusId").reset_index(drop=True)\n\n\ndef _style(ax, title: str, xlabel: str, ylabel: str) -> None:\n    ax.set_title(title, loc="left", fontsize=12, pad=8)\n    ax.set_xlabel(xlabel)\n    ax.set_ylabel(ylabel)\n    ax.spines["top"].set_visible(False)\n    ax.spines["right"].set_visible(False)\n    ax.grid(axis="x", color="#e6e6e6")\n    ax.set_axisbelow(True)\n\n\ndef front_row_conversion(\n    driver_race: pd.DataFrame,\n    circuits: pd.DataFrame,\n    qualifying: pd.DataFrame,\n) -> dict[str, pd.DataFrame]:\n    """Question 1. Front row means grid 1 or 2. Qualifying position is kept separate."""\n    entries = driver_race.loc[driver_race["year"] <= TEST_LAST_YEAR, [\n        "raceId", "driverId", "year", "circuitId", "grid", "podium", "race_name",\n    ]].copy()\n    entries = entries.merge(\n        circuits[["circuitId", "name", "country"]].rename(columns={"name": "circuit_name"}),\n        on="circuitId",\n        how="left",\n        validate="many_to_one",\n    )\n    quali = qualifying[["raceId", "driverId", "position"]].rename(columns={"position": "quali_position"})\n    entries = entries.merge(quali, on=["raceId", "driverId"], how="left", validate="one_to_one")\n    entries["podium"] = entries["podium"].astype(int)\n    entries["era"] = np.where(entries["year"] >= 2014, "2014-2024", "1950-2013")\n    entries["grid_front"] = entries["grid"].isin([1, 2])\n    entries["quali_front"] = entries["quali_position"].isin([1, 2])\n\n    def _rank(front_rows: pd.DataFrame, era_name: str) -> pd.DataFrame:\n        grouped = (\n            front_rows.groupby(["circuit_name", "country"], as_index=False)\n            .agg(front_row_starts=("podium", "size"), podiums=("podium", "sum"))\n        )\n        grouped = grouped.loc[grouped["front_row_starts"] >= MIN_FRONT_ROW_STARTS].copy()\n        grouped["podium_rate"] = grouped["podiums"] / grouped["front_row_starts"]\n        grouped["era"] = era_name\n        grouped["rank"] = grouped["podium_rate"].rank(ascending=False, method="min").astype(int)\n        return grouped.sort_values(["rank", "circuit_name"]).reset_index(drop=True)\n\n    grid_known = entries.loc[entries["grid"].notna()]\n    pre = _rank(grid_known.loc[(grid_known["era"] == "1950-2013") & grid_known["grid_front"]], "1950-2013")\n    post = _rank(grid_known.loc[(grid_known["era"] == "2014-2024") & grid_known["grid_front"]], "2014-2024")\n    both = pre[["circuit_name", "country", "podium_rate", "rank", "front_row_starts"]].merge(\n        post[["circuit_name", "podium_rate", "rank", "front_row_starts"]],\n        on="circuit_name",\n        suffixes=("_pre", "_post"),\n    )\n    both["ranks_gained"] = both["rank_pre"] - both["rank_post"]\n    both = both.sort_values("ranks_gained", ascending=False).reset_index(drop=True)\n\n    both_known = entries.loc[\n        entries["year"].between(2003, TEST_LAST_YEAR)\n        & entries["grid"].notna()\n        & entries["quali_position"].notna()\n    ].copy()\n    quali_front = both_known.loc[both_known["quali_front"]]\n    moved_back = quali_front.loc[~quali_front["grid_front"]]\n    stayed = quali_front.loc[quali_front["grid_front"]]\n    disagreement = pd.DataFrame([\n        {\n            "group": "Qualified front row and started there",\n            "starts": int(len(stayed)),\n            "podium_rate": float(stayed["podium"].mean()) if len(stayed) else np.nan,\n        },\n        {\n            "group": "Qualified front row but started further back",\n            "starts": int(len(moved_back)),\n            "podium_rate": float(moved_back["podium"].mean()) if len(moved_back) else np.nan,\n        },\n    ])\n    return {"pre_2014": pre, "from_2014": post, "rank_change": both, "qualifying_versus_grid": disagreement}\n\n\ndef plot_front_row(ranked: dict[str, pd.DataFrame]):\n    import matplotlib.pyplot as plt\n\n    fig, axes = plt.subplots(1, 2, figsize=(12.5, 6.2))\n    panels = (("pre_2014", "1950-2013"), ("from_2014", "2014-2024"))\n    for ax, (key, title) in zip(axes, panels):\n        table = ranked[key].nsmallest(12, "rank").sort_values("podium_rate")\n        ax.barh(table["circuit_name"], table["podium_rate"] * 100, color=NAVY)\n        _style(ax, f"Front-row podium rate, {title}", "Percent of front-row starts", "")\n    fig.tight_layout()\n    return fig\n\n\ndef narrate_front_row(ranked: dict[str, pd.DataFrame]) -> str:\n    pre = ranked["pre_2014"].iloc[0]\n    post = ranked["from_2014"].iloc[0]\n    change = ranked["rank_change"]\n    gained = change.iloc[0]\n    lost = change.sort_values("ranks_gained").iloc[0]\n    gap = ranked["qualifying_versus_grid"].set_index("group")\n    stayed = gap.loc["Qualified front row and started there"]\n    moved = gap.loc["Qualified front row but started further back"]\n    return (\n        f"Using the starting grid, the highest front-row conversion before 2014 is "\n        f"{pre[\'circuit_name\']} at {pre[\'podium_rate\'] * 100:.1f}% "\n        f"({int(pre[\'podiums\'])} podiums from {int(pre[\'front_row_starts\'])} front-row starts). "\n        f"From 2014 onward it is {post[\'circuit_name\']} at {post[\'podium_rate\'] * 100:.1f}% "\n        f"({int(post[\'podiums\'])} from {int(post[\'front_row_starts\'])}). "\n        f"Circuits need at least {MIN_FRONT_ROW_STARTS} front-row starts in an era, so a handful of races cannot top the list. "\n        f"Among circuits that clear that bar in both eras, {gained[\'circuit_name\']} gains the most ranks "\n        f"({int(gained[\'rank_pre\'])} to {int(gained[\'rank_post\'])}), and {lost[\'circuit_name\']} loses the most "\n        f"({int(lost[\'rank_pre\'])} to {int(lost[\'rank_post\'])}). "\n        f"Qualifying position is not the same list. In 2003-2024, {int(moved[\'starts\'])} drivers qualified on the front row "\n        f"and started further back. They podium {moved[\'podium_rate\'] * 100:.1f}% of the time, against "\n        f"{stayed[\'podium_rate\'] * 100:.1f}% for the {int(stayed[\'starts\'])} who qualified on the front row and started there. "\n        f"A penalty is why those two columns come apart, so the ranking above uses the grid."\n    )\n\n\ndef home_advantage(driver_race: pd.DataFrame, drivers: pd.DataFrame, circuits: pd.DataFrame) -> pd.DataFrame:\n    """Question 2. Home versus away inside the same starting-grid band."""\n    entries = driver_race.loc[\n        (driver_race["year"] <= TEST_LAST_YEAR) & driver_race["grid"].notna() & (driver_race["grid"] >= 1),\n        ["raceId", "driverId", "circuitId", "grid", "podium", "year"],\n    ].copy()\n    entries = add_home_race(entries, drivers, circuits)\n    bins = [0, 1, 3, 10, 100]\n    labels = ["1", "2-3", "4-10", "11+"]\n    entries["grid_band"] = pd.cut(entries["grid"].astype(float), bins=bins, labels=labels)\n    entries["podium"] = entries["podium"].astype(int)\n    entries["where"] = np.where(entries["home_race"] == 1, "home", "away")\n    table = (\n        entries.groupby(["grid_band", "where"], observed=True)["podium"]\n        .agg(starts="size", podium_rate="mean")\n        .reset_index()\n    )\n    return table\n\n\ndef plot_home(table: pd.DataFrame):\n    import matplotlib.pyplot as plt\n\n    fig, ax = plt.subplots(figsize=(8.5, 4.6))\n    bands = ["1", "2-3", "4-10", "11+"]\n    x = np.arange(len(bands))\n    width = 0.36\n    for offset, where, color in ((-width / 2, "home", CLAY), (width / 2, "away", NAVY)):\n        rates = []\n        for band in bands:\n            row = table.loc[(table["grid_band"].astype(str) == band) & (table["where"] == where)]\n            rates.append(float(row["podium_rate"].iloc[0]) * 100 if len(row) else 0)\n        ax.bar(x + offset, rates, width=width, color=color, label=where)\n    ax.set_xticks(x, bands)\n    _style(ax, "Podium rate at home and away, by starting grid", "Starting grid", "Percent of starts")\n    ax.legend(frameon=False)\n    ax.grid(axis="y", color="#e6e6e6")\n    fig.tight_layout()\n    return fig\n\n\ndef narrate_home(table: pd.DataFrame) -> str:\n    lines = []\n    for band in ["1", "2-3", "4-10", "11+"]:\n        home = table.loc[(table["grid_band"].astype(str) == band) & (table["where"] == "home")].iloc[0]\n        away = table.loc[(table["grid_band"].astype(str) == band) & (table["where"] == "away")].iloc[0]\n        gap = (home["podium_rate"] - away["podium_rate"]) * 100\n        lines.append(\n            f"grid {band}: home {home[\'podium_rate\'] * 100:.1f}% ({int(home[\'starts\'])} starts), "\n            f"away {away[\'podium_rate\'] * 100:.1f}% ({int(away[\'starts\'])} starts), gap {gap:+.1f} points"\n        )\n    return (\n        "The comparison holds the starting position roughly fixed, so a home driver who also qualified at the front is not the whole result. "\n        + " | ".join(lines)\n        + ". Blank grids and grid 0 are left out, because those rows have no starting slot to control for. "\n        "East German nationality is matched to German circuits, and a dual national is at home in either country. "\n        "This is a difference inside a grid band, not a claim that the crowd causes the result."\n    )\n\n\ndef mechanical_retirements(\n    driver_race: pd.DataFrame,\n    status: pd.DataFrame,\n    constructors: pd.DataFrame,\n) -> pd.DataFrame:\n    """Question 3. Mechanical retirements divided by race starts, in two eras."""\n    audit = status_audit(status)\n    entries = driver_race.loc[\n        driver_race["year"].between(2014, TEST_LAST_YEAR),\n        ["raceId", "driverId", "year", "constructorId", "statusId"],\n    ].copy()\n    entries = entries.merge(audit, on="statusId", how="left", validate="many_to_one")\n    entries = entries.merge(\n        constructors[["constructorId", "name"]].rename(columns={"name": "constructor"}),\n        on="constructorId",\n        how="left",\n        validate="many_to_one",\n    )\n    entries = entries.loc[entries["category"] != "did_not_start"].copy()\n    entries["mechanical"] = (entries["category"] == "mechanical").astype(int)\n    entries["era"] = np.where(entries["year"] <= 2021, "2014-2021", "2022-2024")\n    grouped = (\n        entries.groupby(["constructor", "era"], as_index=False)\n        .agg(starts=("mechanical", "size"), mechanical_retirements=("mechanical", "sum"))\n    )\n    grouped["rate"] = grouped["mechanical_retirements"] / grouped["starts"]\n    grouped = grouped.loc[grouped["starts"] >= MIN_CONSTRUCTOR_STARTS].copy()\n    wide = grouped.pivot(index="constructor", columns="era", values=["starts", "mechanical_retirements", "rate"])\n    wide.columns = [f"{value}_{era}" for value, era in wide.columns]\n    wide = wide.reset_index()\n    for column in ("rate_2014-2021", "rate_2022-2024"):\n        if column not in wide.columns:\n            wide[column] = np.nan\n    wide = wide.dropna(subset=["rate_2014-2021", "rate_2022-2024"])\n    wide["rate_change"] = wide["rate_2022-2024"] - wide["rate_2014-2021"]\n    return wide.sort_values("rate_2022-2024", ascending=False).reset_index(drop=True)\n\n\ndef plot_mechanical(table: pd.DataFrame):\n    import matplotlib.pyplot as plt\n\n    ordered = table.sort_values("rate_2022-2024")\n    fig, ax = plt.subplots(figsize=(9, 6))\n    y = np.arange(len(ordered))\n    ax.barh(y - 0.18, ordered["rate_2014-2021"] * 100, height=0.36, color="#9bb4c8", label="2014-2021")\n    ax.barh(y + 0.18, ordered["rate_2022-2024"] * 100, height=0.36, color=CLAY, label="2022-2024")\n    ax.set_yticks(y, ordered["constructor"])\n    _style(ax, "Mechanical retirements per start", "Percent of starts", "")\n    ax.legend(frameon=False)\n    fig.tight_layout()\n    return fig\n\n\ndef plot_feature_evidence(evidence: dict):\n    import matplotlib.pyplot as plt\n\n    fig, axes = plt.subplots(2, 2, figsize=(12, 8))\n    panels = (\n        (axes[0, 0], evidence["grid"], "grid", "Podium rate by grid, training years"),\n        (axes[0, 1], evidence["qualifying"], "quali_position", "Podium rate by qualifying position"),\n        (axes[1, 0], evidence["standings"], "standing_bin", "Podium rate by standings entering the race"),\n        (axes[1, 1], evidence["form"], "form_bin", "Podium rate by previous-5 podium rate"),\n    )\n    for ax, table, key, title in panels:\n        labels = table[key].astype(str)\n        ax.bar(labels, table["podium_rate"] * 100, color=NAVY)\n        ax.set_title(title, loc="left", fontsize=11)\n        ax.tick_params(axis="x", labelrotation=45)\n        ax.spines["top"].set_visible(False)\n        ax.spines["right"].set_visible(False)\n    fig.tight_layout()\n    return fig\n\n\ndef narrate_mechanical(table: pd.DataFrame) -> str:\n    early = table.sort_values("rate_2014-2021", ascending=False).iloc[0]\n    late = table.sort_values("rate_2022-2024", ascending=False).iloc[0]\n    rose = table.sort_values("rate_change", ascending=False).iloc[0]\n    fell = table.sort_values("rate_change").iloc[0]\n    return (\n        f"A mechanical retirement is a start whose status names a failed component. "\n        f"Crashes, driver causes, regulation decisions, and unclassified \'Retired\' rows are not counted. "\n        f"Cars that did not start are out of the denominator. Constructors with fewer than {MIN_CONSTRUCTOR_STARTS} starts in an era are left out of that era, "\n        f"and the comparison below keeps constructors that clear the bar in both. "\n        f"Highest rate in 2014-2021: {early[\'constructor\']} at {early[\'rate_2014-2021\'] * 100:.1f}%. "\n        f"Highest rate in 2022-2024: {late[\'constructor\']} at {late[\'rate_2022-2024\'] * 100:.1f}%. "\n        f"Largest rise: {rose[\'constructor\']} ({rose[\'rate_change\'] * 100:+.1f} points). "\n        f"Largest fall: {fell[\'constructor\']} ({fell[\'rate_change\'] * 100:+.1f} points)."\n    )\n'

import os
import sys
import types
from pathlib import Path

import matplotlib.pyplot as plt

import numpy as np
import pandas as pd


def _install(alias, source):
    module = types.ModuleType(alias)
    module.__dict__["__name__"] = alias
    sys.modules[alias] = module
    sys.modules["src." + alias] = module
    setattr(src_pkg, alias, module)
    exec(compile(source, alias + ".py", "exec"), module.__dict__)


src_pkg = types.ModuleType("src")
src_pkg.__path__ = []
sys.modules["src"] = src_pkg
_install("f1_clean", CLEAN_SOURCE)
_install("f1_features", FEATURES_SOURCE)
_install("f1_questions", QUESTIONS_SOURCE)

from src.f1_clean import (
    ASSIGNMENT_LAST_YEAR,
    SENTINEL,
    TABLES,
    clean_tables,
    collapse_entries,
    load_raw,
    raw_hashes,
    replace_sentinels,
)


def find_raw_dir() -> Path:
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        search_roots = [kaggle_input, kaggle_input / "formula-1-world-championship-1950-2020"]
    else:
        cwd = Path.cwd().resolve()
        search_roots = [cwd]
        if (cwd.parent / "data" / "raw" / "results.csv").exists():
            search_roots.append(cwd.parent)

    folders = []
    seen = set()

    def add_folder(folder: Path) -> None:
        try:
            resolved = folder.resolve()
        except OSError:
            return
        if not resolved.is_dir():
            return
        key = str(resolved).lower()
        if key in seen or resolved.name.lower() in {"cleaned", "processed", "audit", "not_in_brief", "working"}:
            return
        seen.add(key)
        folders.append(resolved)

    for root in search_roots:
        if not root.exists():
            continue
        add_folder(root)
        try:
            children = list(root.iterdir())
        except OSError as error:
            print("Could not list", root, error)
            children = []
        for child in children:
            add_folder(child)
            try:
                for grandchild in child.resolve().iterdir():
                    add_folder(grandchild)
            except OSError:
                continue
        for dirpath, _, _ in os.walk(root, followlinks=True):
            add_folder(Path(dirpath))

    required = ("results.csv", "races.csv", "drivers.csv", "status.csv")
    found = []
    for folder in folders:
        names = {path.name.lower(): path for path in folder.glob("*.csv")}
        if all(name in names for name in required):
            found.append(folder)
    if not found:
        listing = []
        for root in search_roots:
            if not root.exists():
                listing.append(f"{root} does not exist")
                continue
            try:
                entries = list(root.iterdir())
            except OSError as error:
                listing.append(f"{root} could not be listed: {error}")
                continue
            if not entries:
                listing.append(f"{root} is empty")
            for entry in entries:
                listing.append(f"{entry} -> {entry.resolve()}")
        detail = " | ".join(listing) if listing else "no input folder was visible"
        raise FileNotFoundError(
            "No Formula 1 CSVs were found in the notebook input. "
            "On the right, open Input, click Add Data, and add "
            "Formula 1 World Championship (1950 - 2024) by Vopani. "
            "Wait until that name appears under Input, then Run All again. "
            "What was visible: " + detail
        )

    def score(folder: Path) -> tuple:
        present = sum((folder / f"{name}.csv").exists() for name in TABLES)
        return (present, -len(folder.parts))

    chosen = max(found, key=score)
    print("Dataset folder:", chosen)
    for path in sorted(chosen.glob("*.csv")):
        print(" ", path.name)
    return chosen


RAW_DIR = find_raw_dir()
missing = [name for name in TABLES if not (RAW_DIR / f"{name}.csv").exists()]
if missing:
    raise FileNotFoundError(f"Missing tables in {RAW_DIR}: {missing}")

if Path("/kaggle/working").exists():
    WORK_DIR = Path("/kaggle/working")
else:
    WORK_DIR = Path.cwd().resolve()
    if not (WORK_DIR / "data").exists() and (WORK_DIR.parent / "data").exists():
        WORK_DIR = WORK_DIR.parent
    WORK_DIR = WORK_DIR / "kaggle_output"

PATHS = {
    "root": WORK_DIR,
    "raw": RAW_DIR,
    "cleaned": WORK_DIR / "cleaned",
    "processed": WORK_DIR / "processed",
    "audit": WORK_DIR / "audit",
    "figures": WORK_DIR / "figures",
}
for folder in ("cleaned", "processed", "audit", "figures"):
    PATHS[folder].mkdir(parents=True, exist_ok=True)
HASHES_AT_START = raw_hashes(PATHS["raw"])

pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 40)
pd.set_option("display.width", 120)

NAVY = "#1f4e79"
CLAY = "#c46b2c"
TEAL = "#2f6f6a"


def styled(ax, title, xlabel, ylabel, grid="y"):
    ax.set_title(title, loc="left", fontsize=13, pad=10)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.grid(axis=grid, color="#e6e6e6")
    ax.set_axisbelow(True)


def save(fig, name):
    fig.tight_layout()
    fig.savefig(PATHS["figures"] / name, dpi=140, bbox_inches="tight", facecolor="white")
    plt.show()


raw = load_raw(PATHS["raw"])
print(f"Loaded {len(raw)} tables from {PATHS['raw']}")
print(f"Writing outputs to {WORK_DIR}")


FileNotFoundError: No Formula 1 CSVs were found in the notebook input. On the right, open Input, click Add Data, and add Formula 1 World Championship (1950 - 2024) by Vopani. Wait until that name appears under Input, then Run All again. What was visible: /kaggle/input is empty | /kaggle/input/formula-1-world-championship-1950-2020 does not exist

## 2. What is in the files

In [ ]:
inventory = pd.DataFrame([
    {"table": name, "rows": len(frame), "columns": len(frame.columns)}
    for name, frame in raw.items()
])
display(inventory)

races_raw = replace_sentinels(raw["races"]).copy()
races_raw["year"] = pd.to_numeric(races_raw["year"])
print(
    f"Seasons in races: {int(races_raw['year'].min())}–{int(races_raw['year'].max())}. "
    f"Distinct seasons: {races_raw['year'].nunique()}."
)
print(f"Assignment window named in the brief: 1950–{ASSIGNMENT_LAST_YEAR}.")

In [ ]:
results_year = replace_sentinels(raw["results"])[["raceId"]].merge(
    races_raw[["raceId", "year"]], on="raceId", how="left"
)
volume = pd.DataFrame({
    "races": races_raw.groupby("year").size(),
    "result_rows": results_year.groupby("year").size(),
}).fillna(0)

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(volume.index, volume["races"], color=TEAL, linewidth=2, label="Races")
ax.plot(volume.index, volume["result_rows"], color=NAVY, linewidth=2, label="Result rows")
styled(ax, "Race calendar and result rows by season", "Season", "Count")
ax.legend(frameon=False)
save(fig, "01_season_volume.png")

## 3. Sentinel values, before they are converted

In [ ]:
from src.f1_clean import summarize_sentinels

sentinels = summarize_sentinels(raw)
display(sentinels)

status_counts = raw["constructor_results"]["status"].value_counts()
print("constructor_results.status raw values:")
display(status_counts.to_frame("rows"))

plot_df = sentinels.sort_values("sentinel_share")
fig, ax = plt.subplots(figsize=(10, max(6, 0.28 * len(plot_df))))
ax.barh(
    plot_df["table"] + "." + plot_df["column"],
    plot_df["sentinel_share"] * 100,
    color=NAVY,
)
styled(
    ax,
    "Share of cells holding the \\N sentinel",
    "Percent of rows",
    "",
    grid="x",
)
save(fig, "02_sentinel_share.png")

In [ ]:
results_status = replace_sentinels(raw["results"]).merge(
    replace_sentinels(raw["status"]), on="statusId", how="left"
)
top_status = results_status["status"].value_counts().head(12).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(top_status.index, top_status.values, color=NAVY)
styled(ax, "Twelve most common finishing statuses", "Result rows", "", grid="x")
save(fig, "03_status_counts.png")
print(f"Distinct statuses used in results: {results_status['status'].nunique()}")

## 4. Keys before cleaning

In [ ]:
from src.f1_clean import audit_keys

keys_before = audit_keys(raw)
failed_before = keys_before.loc[~keys_before["passed"]].copy()
print(f"{int(keys_before['passed'].sum())} of {len(keys_before)} checks passed before cleaning.")
display(failed_before)

## 5. Repeated driver-race rows

In [ ]:
results_view = replace_sentinels(raw["results"]).copy()
for column in ["grid", "positionOrder", "laps", "points"]:
    results_view[column] = pd.to_numeric(results_view[column])

drivers_view = replace_sentinels(raw["drivers"]).copy()
drivers_view["driver"] = drivers_view["forename"] + " " + drivers_view["surname"]

repeated = results_view.loc[results_view.duplicated(["raceId", "driverId"], keep=False)].copy()
repeated = repeated.merge(races_raw[["raceId", "year", "name"]], on="raceId", how="left")
repeated = repeated.merge(drivers_view[["driverId", "driver"]], on="driverId", how="left")
pairs = repeated.drop_duplicates(["raceId", "driverId"])

print(f"Repeated result rows: {len(repeated)}")
print(f"Driver-race pairs involved: {len(pairs)}")
print(f"Seasons involved: {int(pairs['year'].min())}–{int(pairs['year'].max())}")

year_counts = pairs.groupby("year").size().reindex(range(1950, 1979), fill_value=0)
fig, ax = plt.subplots(figsize=(11, 4.2))
ax.bar(year_counts.index, year_counts.values, color=NAVY, width=0.8)
styled(ax, "Driver-race pairs that appear more than once", "Season", "Pairs")
save(fig, "04_duplicate_pairs_by_year.png")

In [ ]:
french_id = races_raw.loc[
    (races_raw["year"] == 1951) & (races_raw["name"] == "French Grand Prix"),
    "raceId",
].iloc[0]
french = repeated.loc[repeated["raceId"] == french_id].copy()
shared_numbers = french.groupby("number")["driverId"].nunique()
shared_numbers = shared_numbers[shared_numbers > 1].index
french_shared = french.loc[french["number"].isin(shared_numbers), [
    "driver", "number", "grid", "positionText", "positionOrder", "laps", "points",
]].sort_values(["number", "positionOrder", "driver"])
display(french_shared)

In [ ]:
late = repeated.loc[repeated["year"] >= 1965, [
    "year", "name", "driver", "constructorId", "grid", "positionText", "positionOrder", "laps",
]].sort_values(["year", "driver", "positionOrder"])
print(f"Repeated rows after 1964: {len(late)}")
display(late)

status_view = replace_sentinels(raw["status"])
late_status = repeated.loc[repeated["year"] >= 1965, ["statusId"]].merge(status_view, on="statusId")
print("Statuses on those rows:")
display(late_status["status"].value_counts().to_frame("rows"))

## 6. What “podium” means in this file

In [ ]:
results_view["brief_podium"] = results_view["positionOrder"] <= 3
results_view["official_podium"] = results_view["positionText"].isin(["1", "2", "3"])
brief_only = results_view["brief_podium"] & ~results_view["official_podium"]
official_only = results_view["official_podium"] & ~results_view["brief_podium"]
print(f"Brief podium rows: {int(results_view['brief_podium'].sum())}")
print(f"Official podium rows: {int(results_view['official_podium'].sum())}")
print(f"In the brief label only: {int(brief_only.sum())}")
print(f"In the official label only: {int(official_only.sum())}")

conflict_ids = results_view.loc[brief_only, "raceId"].unique()
conflict_race = results_view.loc[results_view["raceId"].isin(conflict_ids)].copy()
conflict_race = conflict_race.merge(drivers_view[["driverId", "driver"]], on="driverId", how="left")
conflict_race = conflict_race.merge(races_raw[["raceId", "year", "name"]], on="raceId", how="left")
conflict_race = conflict_race.merge(status_view, on="statusId", how="left")
show_cols = ["year", "name", "driver", "grid", "positionText", "positionOrder", "points", "status"]
display(
    conflict_race.sort_values(["raceId", "positionOrder"])[show_cols]
    .head(8)
)

## 7. Coverage the later feature work has to respect

In [ ]:
result_race_ids = set(results_view["raceId"])
unraced = races_raw.loc[~races_raw["raceId"].isin(result_race_ids), ["year", "round", "name", "date"]]
unraced = unraced.sort_values(["year", "round"])
print(f"Scheduled races with no result rows: {len(unraced)}")
display(unraced)

quali_ids = set(replace_sentinels(raw["qualifying"])["raceId"])
completed = races_raw.loc[races_raw["raceId"].isin(result_race_ids)].copy()
completed["has_qualifying"] = completed["raceId"].isin(quali_ids)
coverage = completed.groupby("year")["has_qualifying"].mean()
print(f"First season present in qualifying: {int(coverage[coverage > 0].index.min())}")

fig, ax = plt.subplots(figsize=(11, 4.2))
ax.plot(coverage.index, coverage.values * 100, color=NAVY, linewidth=2)
ax.axvspan(1994, 2002.5, color="#f3e3d3", zorder=0)
styled(ax, "Completed races that have a qualifying table", "Season", "Percent of races")
ax.set_ylim(-5, 105)
save(fig, "05_qualifying_coverage.png")

In [ ]:
grid_null = results_view.loc[results_view["grid"].isna(), ["raceId"]].merge(
    races_raw[["raceId", "year", "name", "date"]], on="raceId"
)
print("Rows with a null grid, by race:")
display(grid_null.groupby(["year", "name", "date"]).size().rename("null_grid_rows").reset_index())
print(f"Rows with grid 0: {int((results_view['grid'] == 0).sum())}")

debut = results_view.merge(races_raw[["raceId", "year"]], on="raceId").groupby("driverId")["year"].min()
driver_codes = drivers_view.merge(debut.rename("debut"), left_on="driverId", right_index=True, how="inner")
driver_codes["decade"] = (driver_codes["debut"] // 10 * 10).astype(int)
driver_codes["code_missing"] = driver_codes["code"].isna()
driver_codes["number_missing"] = driver_codes["number"].isna()
code_by_decade = driver_codes.groupby("decade").agg(
    drivers=("driverId", "size"),
    code_missing=("code_missing", "mean"),
    number_missing=("number_missing", "mean"),
)
code_by_decade[["code_missing", "number_missing"]] = code_by_decade[["code_missing", "number_missing"]] * 100
display(code_by_decade.round(1))

fig, ax = plt.subplots(figsize=(10, 4.4))
x = np.arange(len(code_by_decade))
width = 0.38
ax.bar(x - width / 2, code_by_decade["code_missing"], width=width, color=NAVY, label="3-letter code missing")
ax.bar(x + width / 2, code_by_decade["number_missing"], width=width, color=CLAY, label="Permanent number missing")
styled(ax, "Missing modern driver identifiers by decade of debut", "Decade of first result", "Percent of drivers")
ax.set_xticks(x, [str(int(year)) + "s" for year in code_by_decade.index], rotation=0)
ax.set_ylim(0, 105)
ax.legend(frameon=False)
save(fig, "06_driver_identifier_missingness.png")

In [ ]:
standings = replace_sentinels(raw["driver_standings"]).copy()
standings["points"] = pd.to_numeric(standings["points"])
unranked = standings.loc[standings["position"].isna()].merge(
    races_raw[["raceId", "year", "name"]], on="raceId"
)
print(f"Driver-standings rows with no position: {len(unranked)}")
display(
    unranked.groupby(["year", "name", "positionText"])
    .size()
    .rename("rows")
    .reset_index()
)

## 8. The cleaning rules

In [ ]:
result = clean_tables(raw)

print("Coercion failures (empty means every present value cast successfully):")
display(result.coercion_failures)

print("Parsed strings against the millisecond columns already in the file:")
display(result.time_checks)

print(f"Disqualified constructor results: {int(result.cleaned['constructor_results']['disqualified'].sum())}")

print("Missing-value method for every column that had a blank marker:")
display(result.missing_policy)
left_blank = result.missing_policy.loc[result.missing_policy["method"] == "leave missing", "values_filled"]
assert left_blank.eq(0).all()

In [ ]:
q1 = result.cleaned["qualifying"][["raceId", "driverId", "q1", "q1_seconds"]].dropna(subset=["q1_seconds"])
ordinary = q1.loc[q1["q1_seconds"] <= 180, "q1_seconds"]
odd = q1.loc[q1["q1_seconds"] > 180].merge(
    result.cleaned["races"][["raceId", "year", "name"]], on="raceId"
).merge(
    result.cleaned["drivers"][["driverId", "forename", "surname"]], on="driverId"
)
fig, ax = plt.subplots(figsize=(10, 4.2))
ax.hist(ordinary, bins=40, color=NAVY)
styled(
    ax,
    f"Parsed Q1 times, excluding {len(odd)} value above 3 minutes. Median {ordinary.median():.1f}s",
    "Q1 (seconds)",
    "Qualifying rows",
)
save(fig, "07_q1_seconds.png")
print("Qualifying times longer than 3 minutes, left as parsed from the source string:")
display(odd[["year", "name", "forename", "surname", "q1", "q1_seconds"]])

sample_times = result.cleaned["results"].loc[
    result.cleaned["results"]["winner_time_seconds"].notna()
    | result.cleaned["results"]["gap_seconds"].notna(),
    ["time", "winner_time_seconds", "gap_seconds", "elapsed_seconds"],
].head(6)
display(sample_times)

In [ ]:
keys_after = result.keys_after
failed_after = keys_after.loc[~keys_after["passed"]].copy()
print(f"{int(keys_after['passed'].sum())} of {len(keys_after)} checks passed after cleaning.")
display(failed_after)
print("Lap-time key repair:")
display(result.lap_conflict_summary)

## 9. Which row is kept

In [ ]:
comparison = result.rule_comparison
print(f"Repeated pairs compared: {len(comparison)}")
print(f"Same resultId: {int(comparison['same_row'].sum())}")
print(f"Same positionOrder: {int(comparison['same_position'].sum())}")
print(f"Same podium label: {int(comparison['same_podium'].sum())}")
print(f"Disagreements: {int((~comparison['same_row']).sum())}")

toy = pd.DataFrame([
    {"raceId": 1, "driverId": 7, "resultId": 10, "positionOrder": 8, "laps": 70},
    {"raceId": 1, "driverId": 7, "resultId": 11, "positionOrder": 2, "laps": 12},
])
toy_compare = pd.DataFrame({
    "rule": ["best_classification", "most_laps"],
    "kept_resultId": [
        int(collapse_entries(toy, "best_classification").iloc[0]["resultId"]),
        int(collapse_entries(toy, "most_laps").iloc[0]["resultId"]),
    ],
})
display(toy)
display(toy_compare)

## 10. The driver-race table, and what cleaning changed

In [ ]:
driver_race = result.driver_race
print(f"Result rows: {len(result.cleaned['results'])}")
print(f"Driver-race rows: {len(driver_race)}")
print(f"Rows removed by the collapse: {len(result.cleaned['results']) - len(driver_race)}")
print(f"Unique (raceId, driverId): {not driver_race.duplicated(['raceId', 'driverId']).any()}")
print(f"Shared entries: {int(driver_race['shared_entry'].sum())}")
print(f"Shared entries with more than one constructor: {int((driver_race['n_constructors'] > 1).sum())}")
print(f"Podium definition conflicts: {int(driver_race['podium_definition_conflict'].sum())}")
print(
    f"Rows inside the brief's 1950–{ASSIGNMENT_LAST_YEAR} window: "
    f"{int(driver_race['in_assignment_window'].sum())} of {len(driver_race)}"
)

before = results_view[["raceId", "positionOrder"]].merge(
    races_raw[["raceId", "year"]], on="raceId", how="left"
)
before["podium"] = before["positionOrder"] <= 3
before["decade"] = (before["year"] // 10 * 10).astype(int)
after = driver_race[["raceId", "year", "podium"]].copy()
after["decade"] = (after["year"] // 10 * 10).astype(int)

rate = pd.DataFrame({
    "before": before.groupby("decade")["podium"].mean() * 100,
    "after": after.groupby("decade")["podium"].mean() * 100,
})
field = (
    after.groupby(["decade", "raceId"]).size()
    .groupby("decade").median()
    .rename("median_starters")
)
display(rate.round(2).join(field))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
x = np.arange(len(rate))
width = 0.38
axes[0].bar(x - width / 2, rate["before"], width=width, color="#9bb4c8", label="Before collapse")
axes[0].bar(x + width / 2, rate["after"], width=width, color=NAVY, label="After collapse")
styled(axes[0], "Podium rate by decade", "Decade", "Percent of rows")
axes[0].set_xticks(x, [str(int(year)) + "s" for year in rate.index], rotation=45)
axes[0].legend(frameon=False)

axes[1].bar(x, field.values, color=TEAL)
styled(axes[1], "Median starters per race", "Decade", "Drivers")
axes[1].set_xticks(x, [str(int(year)) + "s" for year in field.index], rotation=45)
save(fig, "08_podium_rate_before_after.png")

In [ ]:
balance = driver_race["podium"].value_counts().rename({False: "Not podium", True: "Podium"})
balance_share = driver_race["podium"].mean()
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(balance.index, balance.values, color=[NAVY, CLAY])
styled(ax, f"Class balance on driver_race ({balance_share * 100:.1f}% podium)", "", "Rows")
save(fig, "09_class_balance.png")

multi_podium = driver_race.groupby("raceId")["podium"].sum()
print("Races by how many drivers carry the podium label:")
display(multi_podium.value_counts().sort_index().rename("races").to_frame())

In [ ]:
grid = driver_race.dropna(subset=["grid"]).copy()
grid["grid"] = grid["grid"].astype(int)
grid_rates = grid.groupby("grid")["podium"].agg(rate="mean", starts="size")
plotted = grid_rates.loc[grid_rates["starts"] >= 50].copy()
omitted = int(grid_rates.loc[grid_rates["starts"] < 50, "starts"].sum())

fig, ax = plt.subplots(figsize=(11, 4.4))
colors = [CLAY if grid_pos == 0 else NAVY for grid_pos in plotted.index]
ax.bar(plotted.index.astype(str), plotted["rate"] * 100, color=colors)
styled(ax, "Podium rate by starting grid", "Grid", "Percent of starts")
save(fig, "10_podium_rate_by_grid.png")
grid_zero = driver_race.loc[driver_race["grid"] == 0, "podium"]
print(f"Grid 0 podium rate: {grid_zero.mean():.1%} on {len(grid_zero)} starts")
print(f"Starts on grids with fewer than 50 starts, omitted from the chart: {omitted}")
print(f"Starts with a null grid, also omitted: {int(driver_race['grid'].isna().sum())}")

## 11. Handoff

In [ ]:
from src.f1_clean import DRIVER_RACE_DICTIONARY, save_outputs

dictionary = pd.DataFrame(DRIVER_RACE_DICTIONARY, columns=["column", "role", "meaning"])
display(dictionary)

kept_example = result.duplicate_rows.loc[result.duplicate_rows["raceId"] == int(french_id)].copy()
name_map = (
    drivers_view.assign(driverId=pd.to_numeric(drivers_view["driverId"]))
    .set_index("driverId")["driver"]
)
kept_example["driver"] = kept_example["driverId"].map(name_map)
shared_number_ids = pd.to_numeric(pd.Index(shared_numbers))
display(
    kept_example.loc[kept_example["number"].isin(shared_number_ids), [
        "driver", "number", "grid", "positionText", "positionOrder", "laps", "kept",
    ]].sort_values(["number", "positionOrder", "driver"])
)

assert result.coercion_failures.empty
time_gap = result.time_checks.loc[~(result.time_checks["max_abs_seconds"] < 0.01)]
if time_gap.empty:
    print("Parsed times match the millisecond columns within 0.01 seconds.")
else:
    print("These source columns disagree with the millisecond values by 0.01 seconds or more. Both numbers are kept:")
    display(time_gap)
assert not driver_race.duplicated(["raceId", "driverId"]).any()
assert not result.cleaned["lap_times"].duplicated(["raceId", "driverId", "lap"]).any()
assert HASHES_AT_START == raw_hashes(PATHS["raw"])

save_outputs(result, PATHS)
print("Wrote cleaned tables, driver_race, and audit files.")
print(f"Raw files unchanged: {HASHES_AT_START == raw_hashes(PATHS['raw'])}")

## 12. Features that exist before the race

In [ ]:
from src.f1_features import (
    FEATURE_COLUMNS,
    LEAKAGE_COLUMNS,
    build_model_frame,
    feature_dictionary_frame,
    feature_evidence,
    modeling_sample,
)
from src.f1_questions import plot_feature_evidence

model = build_model_frame(result.driver_race, result.cleaned)
sample = modeling_sample(model)
display(feature_dictionary_frame())
print("Rows in the modeling window:", len(sample))
print(sample["split"].value_counts().to_string())
assert not any(column in model.columns for column in LEAKAGE_COLUMNS)
assert not sample.duplicated(["raceId", "driverId"]).any()

evidence = feature_evidence(model)
for name, table in evidence.items():
    print(name)
    display(table)
save(plot_feature_evidence(evidence), "11_feature_evidence.png")
model.to_csv(PATHS["processed"] / "model_frame.csv", index=False)
feature_dictionary_frame().to_csv(PATHS["processed"] / "model_feature_dictionary.csv", index=False)
print("Wrote model_frame.csv")

## 13. Which circuits convert a front-row start into a podium?

In [ ]:
from src.f1_questions import (
    front_row_conversion,
    home_advantage,
    mechanical_retirements,
    narrate_front_row,
    narrate_home,
    narrate_mechanical,
    plot_front_row,
    plot_home,
    plot_mechanical,
    status_audit,
)

ranked = front_row_conversion(result.driver_race, result.cleaned["circuits"], result.cleaned["qualifying"])
print(narrate_front_row(ranked))
display(ranked["pre_2014"].head(8))
display(ranked["from_2014"].head(8))
display(ranked["rank_change"].head(8))
display(ranked["qualifying_versus_grid"])
save(plot_front_row(ranked), "12_front_row_conversion.png")

## 14. Do drivers podium more often at home once the start slot is held fixed?

In [ ]:
home = home_advantage(result.driver_race, result.cleaned["drivers"], result.cleaned["circuits"])
print(narrate_home(home))
display(home)
save(plot_home(home), "13_home_by_grid.png")

## 15. Which constructors retire for mechanical reasons?

In [ ]:
audit = status_audit(result.cleaned["status"])
print(audit["category"].value_counts().to_string())
mechanical = mechanical_retirements(
    result.driver_race, result.cleaned["status"], result.cleaned["constructors"]
)
print(narrate_mechanical(mechanical))
display(mechanical)
save(plot_mechanical(mechanical), "14_mechanical_retirements.png")
audit.to_csv(PATHS["audit"] / "status_categories.csv", index=False)

In [ ]:
from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42

def to_float(series):
    as_object = series.astype("object")
    return pd.to_numeric(as_object.where(series.notna(), np.nan), errors="coerce").astype("float64")

def prepare_features(frame):
    X = pd.DataFrame({c: to_float(frame[c]) for c in FEATURE_COLUMNS}, index=frame.index)
    field_size = frame.groupby("raceId")["driverId"].transform("size").astype("float64")
    pit_lane = X["grid"].eq(0)
    X["pit_lane_start"] = pit_lane.astype(float)
    X.loc[pit_lane, "grid"] = field_size[pit_lane]
    return X

def make_pipeline(estimator, scale):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", clone(estimator)))
    return Pipeline(steps)

modeling = sample.sort_values(["race_date", "raceId", "driverId"]).reset_index(drop=True)
X_all, y_all = prepare_features(modeling), modeling["podium"].astype(int)
MODEL_FEATURES = list(X_all.columns)
masks = {s: modeling["split"].eq(s).to_numpy() for s in ("train", "validation", "test")}
X_train, y_train = X_all[masks["train"]], y_all[masks["train"]]
X_val, y_val = X_all[masks["validation"]], y_all[masks["validation"]]
X_test, y_test = X_all[masks["test"]], y_all[masks["test"]]

display(pd.DataFrame({
    s: {"years": f"{int(modeling.loc[m, 'year'].min())}-{int(modeling.loc[m, 'year'].max())}",
        "rows": int(m.sum()), "podium_rate_%": round(y_all[m].mean() * 100, 2)}
    for s, m in masks.items()}).T)

In [ ]:
CANDIDATES = {
    "Logistic regression": (LogisticRegression(max_iter=3000), True,
        [{"C": 0.01}, {"C": 0.1}, {"C": 1.0}, {"C": 10.0}]),
    "Random forest": (RandomForestClassifier(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE), False,
        [{"max_depth": 4, "min_samples_leaf": 20}, {"max_depth": 8, "min_samples_leaf": 20},
         {"max_depth": None, "min_samples_leaf": 50}]),
    "Gradient boosting": (HistGradientBoostingClassifier(random_state=RANDOM_STATE, early_stopping=False), False,
        [{"learning_rate": 0.05, "max_depth": 3, "max_iter": 300},
         {"learning_rate": 0.1, "max_leaf_nodes": 15, "max_iter": 150},
         {"learning_rate": 0.03, "max_depth": 4, "max_iter": 500, "l2_regularization": 1.0}]),
    "Shallow NN (MLP)": (MLPClassifier(max_iter=500, early_stopping=True, validation_fraction=0.15,
                                       n_iter_no_change=20, random_state=RANDOM_STATE), True,
        [{"hidden_layer_sizes": (16,), "alpha": 1e-3}, {"hidden_layer_sizes": (32, 16), "alpha": 1e-3},
         {"hidden_layer_sizes": (64, 32), "alpha": 1e-2}]),
}

trials, fitted = [], {}
for family, (estimator, scale, grid_params) in CANDIDATES.items():
    for params in grid_params:
        pipe = make_pipeline(estimator, scale).set_params(**{f"model__{k}": v for k, v in params.items()})
        pipe.fit(X_train, y_train)
        p_tr, p_va = pipe.predict_proba(X_train)[:, 1], pipe.predict_proba(X_val)[:, 1]
        trials.append({"model": family, "params": str(params),
                       "train_PR-AUC": average_precision_score(y_train, p_tr),
                       "val_PR-AUC": average_precision_score(y_val, p_va),
                       "val_ROC-AUC": roc_auc_score(y_val, p_va)})
        fitted[(family, str(params))] = pipe

trials = pd.DataFrame(trials)
trials["overfit_gap"] = trials["train_PR-AUC"] - trials["val_PR-AUC"]
display(trials.round(4))
trials.to_csv(PATHS["audit"] / "model_search_trials.csv", index=False)

fig, ax = plt.subplots(figsize=(11, 4.6))
x = np.arange(len(trials))
ax.bar(x - 0.2, trials["train_PR-AUC"], 0.4, color="#9bb4c8", label="train")
ax.bar(x + 0.2, trials["val_PR-AUC"], 0.4, color=NAVY, label="validation")
ax.set_xticks(x, trials["model"] + "\n" + trials["params"].str.slice(0, 32), rotation=75, fontsize=7)
styled(ax, "Hyperparameter search: PR-AUC on train vs validation", "", "PR-AUC")
ax.legend(frameon=False)
save(fig, "15_hyperparameter_search.png")

In [ ]:
def best_f1_threshold(y_true, proba):
    precision, recall, thr = precision_recall_curve(y_true, proba)
    f1 = 2 * precision * recall / np.clip(precision + recall, 1e-12, None)
    return float(thr[int(np.nanargmax(f1[:-1]))])

def evaluate(y_true, proba, threshold):
    pred = (proba >= threshold).astype(int)
    return {"ROC-AUC": roc_auc_score(y_true, proba), "PR-AUC": average_precision_score(y_true, proba),
            "F1": f1_score(y_true, pred, zero_division=0),
            "Precision": precision_score(y_true, pred, zero_division=0),
            "Recall": recall_score(y_true, pred, zero_division=0)}

best_rows = trials.loc[trials.groupby("model")["val_PR-AUC"].idxmax()]
final_pipelines = {r.model: fitted[(r.model, r.params)] for r in best_rows.itertuples()}
display(best_rows[["model", "params", "val_PR-AUC"]])

split_data = {"train": (X_train, y_train), "validation": (X_val, y_val), "test": (X_test, y_test)}
thresholds = {n: best_f1_threshold(y_val, p.predict_proba(X_val)[:, 1]) for n, p in final_pipelines.items()}
probas = {n: {s: p.predict_proba(X)[:, 1] for s, (X, _) in split_data.items()} for n, p in final_pipelines.items()}

# Naive floor: rank by grid, predict podium when grid <= 3
probas["Grid rule (grid <= 3)"] = {s: (-X["grid"]).fillna(-X["grid"].max()).to_numpy() for s, (X, _) in split_data.items()}
thresholds["Grid rule (grid <= 3)"] = -3.0

comparison_table = pd.DataFrame([
    {"model": n, "split": s, "threshold": thresholds[n], **evaluate(y, probas[n][s], thresholds[n])}
    for n in probas for s, (_, y) in split_data.items()])
comparison_table.to_csv(PATHS["processed"] / "model_comparison.csv", index=False)

wide = comparison_table.pivot(index="model", columns="split", values=["PR-AUC", "ROC-AUC", "F1", "Precision", "Recall"])
display(wide.reindex(columns=["train", "validation", "test"], level=1).round(3))

In [ ]:
palette = {"Grid rule (grid <= 3)": "#999999", "Logistic regression": NAVY, "Random forest": TEAL,
           "Gradient boosting": CLAY, "Shallow NN (MLP)": "#7a4fa3"}
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for col, s in enumerate(("validation", "test")):
    y = split_data[s][1]
    for n, by in probas.items():
        ls = "--" if n.startswith("Grid") else "-"
        fpr, tpr, _ = roc_curve(y, by[s]); prec, rec, _ = precision_recall_curve(y, by[s])
        axes[0, col].plot(fpr, tpr, ls, color=palette[n], label=f"{n} ({roc_auc_score(y, by[s]):.3f})")
        axes[1, col].plot(rec, prec, ls, color=palette[n], label=f"{n} ({average_precision_score(y, by[s]):.3f})")
    axes[0, col].plot([0, 1], [0, 1], ":", color="#bbbbbb")
    axes[1, col].axhline(y.mean(), ls=":", color="#bbbbbb")
    styled(axes[0, col], f"ROC, {s}", "False positive rate", "True positive rate")
    styled(axes[1, col], f"Precision-recall, {s}", "Recall", "Precision")
    axes[0, col].legend(frameon=False, fontsize=8); axes[1, col].legend(frameon=False, fontsize=8)
save(fig, "16_roc_pr_curves.png")

fig, axes = plt.subplots(1, len(final_pipelines), figsize=(4 * len(final_pipelines), 3.6))
for ax, n in zip(axes, final_pipelines):
    cm = confusion_matrix(y_test, (probas[n]["test"] >= thresholds[n]).astype(int))
    ax.imshow(cm, cmap="Blues")
    for (i, j), v in np.ndenumerate(cm):
        ax.text(j, i, v, ha="center", va="center")
    ax.set_xticks([0, 1], ["no podium", "podium"]); ax.set_yticks([0, 1], ["no podium", "podium"])
    ax.set_title(f"{n}\nthreshold {thresholds[n]:.2f}", fontsize=10); ax.set_xlabel("predicted")
axes[0].set_ylabel("actual")
save(fig, "17_confusion_matrices_test.png")

val = comparison_table.query("split == 'validation'").set_index("model").drop("Grid rule (grid <= 3)")
chosen = val["PR-AUC"].idxmax()
test = comparison_table.query("split == 'test'").set_index("model")
print(f"Chosen on validation PR-AUC: {chosen}. Test: PR-AUC {test.loc[chosen, 'PR-AUC']:.3f}, "
      f"ROC-AUC {test.loc[chosen, 'ROC-AUC']:.3f}, F1 {test.loc[chosen, 'F1']:.3f}. "
      f"Grid rule test PR-AUC {test.loc['Grid rule (grid <= 3)', 'PR-AUC']:.3f}; no-skill PR-AUC = podium rate {y_test.mean():.3f}.")